# Clinical NLP over Italian Hospital Records

*NLP4DH exam project.* The corpus is 857 pseudonymised hospital encounters, each with three Italian
reports: discharge therapy, admission therapy and anamnesis. The pipeline turns those reports into
structured facts, derives safety alerts from them with symbolic rules, publishes everything as an RDF
graph, and uses a small local language model for extraction and for three neuro-symbolic demonstrations.

**Contents**

1. Setup and data
2. Dataset exploration
3. Drug resources and discharge therapy
4. Admission therapy: few-shot extraction
5. Anamnesis: NER, context and UMLS linking
6. Symbolic alert engine
7. Evaluation
8. RDF graph and SPARQL
9. Local LLM demonstrations
10. Discussion

Each report type gets the method its surface form justifies: a regex for the templated discharge
therapy, a generative model for the heterogeneous admission therapy, and zero-shot NER plus a
rule-based context algorithm for free anamnesis prose. Every clinical assertion in the alerts comes
from an external knowledge base, never from a model.

## 1. Setup and data

In [ ]:
import json
import os
import random
import re
import textwrap
import time
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import ollama
import pandas as pd
import requests
import spacy
import torch
from gliner import GLiNER
from IPython.display import display
from medspacy.context import ConTextRule
from rdflib import Graph, Literal, Namespace, RDF, RDFS, URIRef
from rdflib.plugins.sparql import prepareQuery
from sentence_transformers import SentenceTransformer
from sklearn.metrics import confusion_matrix, f1_score
from spacy import displacy
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()   # the model libraries print download bars that add nothing here

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ANNOTATION_DIR = DATA_DIR / "annotations"
EXTERNAL_DIR = DATA_DIR / "external"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Models. GLiNER tags the anamnesis, E5 ranks UMLS candidates, Qwen is the local instruction model
# used both for admission extraction (Section 4) and for the demonstrations of Section 9.
SPACY_MODEL = "it_core_news_lg"
GLINER_MODEL = "urchade/gliner_multi-v2.1"
EMBEDDER_MODEL = "intfloat/multilingual-e5-small"
LOCAL_LLM_MODEL = "qwen2.5:3b-instruct"

NER_THRESHOLD = 0.45          # GLiNER span confidence

# UMLS Metathesaurus, through the UTS REST API, pinned to one release.
UMLS_VERSION = "2026AA"
UMLS_BASE_URL = "https://uts-ws.nlm.nih.gov/rest"
UMLS_API_KEY = os.getenv("UMLS_API_KEY")
UMLS_CACHE_DIR = EXTERNAL_DIR / "umls_cache"
UMLS_SEMANTIC_GROUP = "Disorders"     # applied server-side, so no mention is compared against a drug concept
UMLS_MAX_CANDIDATES = 15
# Both constants were read off the score distribution of development mentions before the held-out
# gold was opened. Multilingual cosine scores sit in a narrow band, so the margin, not the floor, is
# what decides a tie.
UMLS_COSINE_FLOOR = 0.80
UMLS_COSINE_MARGIN = 0.005
UMLS_LINKED = ("linked_exact", "linked_contextual")

pd.set_option("display.max_colwidth", 90)
print(f"device: {DEVICE} | UMLS release {UMLS_VERSION} | API key configured: {UMLS_API_KEY is not None}")

In [ ]:
with open(DATA_DIR / "pazienti_con_terapia_uscita_testuale.json", encoding="utf-8") as f:
    patients = json.load(f)

patient_by_oid = {p["encOid"]: p for p in patients}

T_ANAMNESIS = "Anamnesi"
T_ADMISSION = "Terapia medica all'ingresso"
T_DISCHARGE = "Terapia alla Dimissione"


def report(patient, report_type):
    """The report record of a given type, or None when the patient has no such report."""
    return next((r for r in patient["referti"] if r["tipo"] == report_type), None)


def report_text(patient, report_type):
    """Free text of a report type, empty string when the report is missing."""
    record = report(patient, report_type)
    return (record["testo"] or "") if record else ""


def wrap(text, width=100):
    """Wrap long free text for readable output, keeping the existing line breaks."""
    return "\n".join(textwrap.fill(line, width=width) if line.strip() else line
                     for line in str(text).splitlines())


print(f"loaded {len(patients)} patients, {sum(len(p['referti']) for p in patients)} reports")

### Development and held-out patients

50 patients are manually annotated. They are split once, **by patient**, with the notebook seed:
20 development patients for every choice that needs data (the few-shot examples of Section 4), and
30 held-out patients that only Section 7 reads. The split is by patient rather than by mention
because mentions from one report share an author, a style and a set of conditions.

In [ ]:
annotated = sorted(int(x) for x in pd.read_csv(ANNOTATION_DIR / "gold_report_manifest.csv").encOid.unique())
shuffled = list(annotated)
random.Random(RANDOM_SEED).shuffle(shuffled)

DEV_PATIENTS = sorted(shuffled[:20])
TEST_PATIENTS = sorted(shuffled[20:])
assert not set(DEV_PATIENTS) & set(TEST_PATIENTS)

print(f"annotated patients: {len(annotated)} -> {len(DEV_PATIENTS)} development, {len(TEST_PATIENTS)} held out")

## 2. Dataset exploration

**Task.** Look at the data before building anything.
**Method.** Report counts and text lengths per report type, then print the cases each later stage has
to survive.
**Output.** A length table, a histogram and a set of worked examples.
**Rationale.** The three report types differ so much in surface form that they justify three
different extraction methods; the examples below are the evidence for that choice.

In [ ]:
lengths = pd.DataFrame([{"report_type": r["tipo"], "characters": len(r["testo"] or "")}
                        for p in patients for r in p["referti"]])

print(f"patients: {len(patients)} | patients with exactly three reports: "
      f"{sum(len(p['referti']) == 3 for p in patients)}")
display(lengths.groupby("report_type").characters.agg(["count", "min", "median", "mean", "max"]).round(0).astype(int))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)

for ax, report_type in zip(axes, [T_ANAMNESIS, T_ADMISSION, T_DISCHARGE]):
    ax.hist(lengths.loc[lengths.report_type == report_type, "characters"],
            bins=40, color="#4C72B0", edgecolor="white")
    ax.set_title(report_type, fontsize=9)
    ax.set_xlabel("characters")

axes[0].set_ylabel("reports")
fig.suptitle("Text length by report type", y=1.03, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
def show(title, text, width=340):
    body = (text[:width] + "…") if len(text) > width else (text or "(empty)")
    print(f"### {title}\n{wrap(body)}\n")


show("Terapia alla Dimissione - quoted template, including a combination product",
     report_text(patients[0], T_DISCHARGE))
show("Terapia medica all'ingresso - semi-structured product list",
     report_text(patients[0], T_ADMISSION))
show("Terapia medica all'ingresso - no home therapy",
     report_text(next(p for p in patients if "essuna terapia" in report_text(p, T_ADMISSION)), T_ADMISSION))
show("Terapia medica all'ingresso - non-drug interventions among the drugs",
     report_text(next(p for p in patients
                      if re.search(r"Ossigeno|NIV|CPAP", report_text(p, T_ADMISSION))), T_ADMISSION))
show("Anamnesi - narrative prose", report_text(patients[0], T_ANAMNESIS))

In [ ]:
# The four phenomena the context stage has to handle, each shown on the sentence that contains it.
def first_sentence_matching(pattern):
    """The first anamnesis sentence in the cohort matching `pattern`, with its patient id."""
    for patient in patients:
        for sentence in re.split(r"(?<=[.!?])\s+", report_text(patient, T_ANAMNESIS)):
            if re.search(pattern, sentence, flags=re.IGNORECASE):
                return patient["encOid"], sentence.strip()
    return None, ""


phenomena = {
    "negation": r"\bnega\b",
    "family history": r"\bfamiliarit",
    "negated family history": r"\bnon\s+familiarit",
    "uncertainty": r"\bsospett[oa]\b|\bverosimile\b",
}

display(pd.DataFrame([{"phenomenon": name, "encOid": oid, "sentence": sentence}
                      for name, pattern in phenomena.items()
                      for oid, sentence in [first_sentence_matching(pattern)]]).set_index("phenomenon"))

| Observation | Consequence |
|---|---|
| Every patient has the three expected reports | Retrieve by type; no missing-report handling |
| *Terapia alla Dimissione* follows a quoted template `Principio attivo (Prodotto forma dose): da assumere …` | Section 3 reads it with a regex |
| *Terapia medica all'ingresso* is short and heterogeneous: product lists, bare free text, empty, or non-drug lines | Section 4 uses few-shot generative extraction |
| *Anamnesi* is long narrative Italian prose with negation, family history and uncertainty | Section 5 uses zero-shot NER plus a context algorithm |

## 3. Drug resources and discharge therapy

**Task.** Build the drug knowledge the rest of the notebook joins on, then extract the discharge
regimen from its templated text.
**Method.** Four small resource tables prepared once, one lexical normaliser, one regex parser.
**Output.** `products`, `ingredients`, `interactions`, `contraindications`, and `prescriptions`.
**Rationale.** Every later stage joins on an identifier rather than on a string, so the identifiers
have to be defined once, in one place, with one meaning each.

### What each identifier is for

| Identifier | Represents | Used for |
|---|---|---|
| `ingredient_key` | canonical local ingredient identity | duplicates, exact allergy/intolerance |
| `atc_code` | product therapeutic/anatomical classification | RDF hierarchy / SPARQL |
| `rxcui` | RxNorm drug concept | MED-RT drug side |
| `ddinter_id` | DDInter drug identifier | drug-drug interactions |
| `umls_cui` | clinical concept | anamnesis condition linking / MED-RT condition side |

Product-level and ingredient-level information are kept apart, because they answer different
questions:

```
AIFA product              canonical ingredient
  product_name              ingredient_key
  ingredients               preferred_label
  atc_code                  rxcui
                            ddinter_id
```

ATC classifies a *registered product* — the same substance carries different codes inhaled, nasal or
topical — so it is a property of the prescription, taken from the product the clinician named, and
left `None` when the product is not in the registry.

### 3.1 Lexical normalisation

Two functions, and no more. `normalize_name` folds case and whitespace; `ingredient_key` additionally
removes the salt, hydrate and dose material that separates a written mention from the registry's name
for the same molecule. `preferred_label` is a display string and is never used as an identity.

In [ ]:
def normalize_name(value):
    """Case- and whitespace-normalised surface form."""
    return re.sub(r"\s+", " ", str(value).strip().lower())


# Counterions, esters and hydration states: two salts of one molecule interact identically, so they
# must reduce to the same key. The trailing-ion pattern is position-aware because these ions can also
# be the active substance ("calcio carbonato" keeps calcio, "atorvastatina calcio" drops it).
SALT_OR_HYDRATE = re.compile(
    r"\b(cloridrato|idrocloruro|besilato|mesilato|maleato|fumarato|emifumarato|tartrato|succinato|"
    r"solfato|acetato|dipropionato|furoato|valerato|nitrato|fosfato|citrato|bromuro|sodico|sodica|"
    r"calcico|potassico|anidro|\w*idrato)\b", re.I)
TRAILING_ION = re.compile(r"\b(?:sale\s+)?di\s+(?:sodio|potassio|calcio|magnesio|zinco)\b"
                          r"|\s+(?:sodio|potassio|calcio|magnesio|zinco)\s*$", re.I)
# The discharge template sometimes leaves posology inside the ingredient field
# ("Pantoprazolo 20 mg 1 cp al mattino"), so the key has to cut it.
DOSE_TAIL = re.compile(r"\b(?:mg|mcg|ml|ui|cp|cpr|cps|gtt|fl|bustin[ae]|die|ore|da assumere)\b.*$", re.I)


def ingredient_key(name):
    """Canonical identity of an active ingredient: the one key every drug rule joins on."""
    text = DOSE_TAIL.sub(" ", normalize_name(name))
    text = re.sub(r"\s+\d+(?:[.,]\d+)?\s*$", " ", text)          # a bare trailing number: a dose, or a polymer grade
    text = TRAILING_ION.sub(" ", SALT_OR_HYDRATE.sub(" ", text))
    return re.sub(r"\s+", " ", text).strip() or normalize_name(name)


assert ingredient_key("ATORVASTATINA CALCIO TRIIDRATO") == "atorvastatina"
assert ingredient_key("Pantoprazolo 20 mg 1 cp al mattino h 7") == "pantoprazolo"
assert ingredient_key("Macrogol 3350") == "macrogol"             # the registry entry is "Macrogol"

### 3.2 Resource tables

`products` and `ingredients` come from the Italian medicines agency (AIFA) registry. `interactions`
is the DDInter 2.0 pairwise table. `contraindications` is MED-RT's `ci_with` relation, already
aligned to UMLS concepts — Section 6 consumes it as a set of pairs and contains no MeSH logic of its
own.

In [ ]:
# AIFA product registry: one row per registered product name, with its active ingredients and, when
# the registry assigns exactly one, its ATC code.
packages = pd.read_parquet(EXTERNAL_DIR / "aifa" / "normalized" / "aifa_confezioni.parquet")
by_product = {}

for name, actives, atc in zip(packages.DENOMINAZIONE, packages.PA_ASSOCIATI, packages.CODICE_ATC):
    if pd.isna(name):
        continue

    entry = by_product.setdefault(normalize_name(name), {"product_name": str(name), "ingredients": set(), "atc": set()})

    if pd.notna(actives):
        entry["ingredients"].update(part.strip() for part in str(actives).split("/") if part.strip())

    if pd.notna(atc):
        entry["atc"].add(str(atc))

products = pd.DataFrame([{"product_key": key,
                          "product_name": entry["product_name"],
                          "ingredients": sorted(entry["ingredients"]),
                          "atc_code": next(iter(entry["atc"])) if len(entry["atc"]) == 1 else None}
                         for key, entry in by_product.items()])

PRODUCT_ATC = dict(zip(products.product_key, products.atc_code))
PRODUCT_INGREDIENTS = dict(zip(products.product_key, products.ingredients))

print(f"AIFA products: {len(products)} | with a single ATC code: {int(products.atc_code.notna().sum())}")
display(products.head(3))

In [ ]:
# Ingredient crosswalk: the canonical ingredient and the two external identifiers the rules need.
# The shipped `lookup_key` is exactly `ingredient_key(preferred_label)`, which the assertion pins.
crosswalk = pd.read_csv(EXTERNAL_DIR / "ingredient_crosswalk.csv")
ingredients = pd.DataFrame({"ingredient_key": crosswalk.lookup_key.astype(str),
                            "preferred_label": crosswalk.preferred_label.astype(str),
                            "rxcui": pd.to_numeric(crosswalk.RXCUI, errors="coerce").astype("Int64"),
                            "ddinter_id": crosswalk.DDINTER.astype("string")})

assert ingredients.ingredient_key.is_unique
assert (ingredients.preferred_label.map(ingredient_key) == ingredients.ingredient_key).all()

PREFERRED_LABEL = dict(zip(ingredients.ingredient_key, ingredients.preferred_label))
KNOWN_INGREDIENTS = set(ingredients.ingredient_key)
RXCUI = {k: int(v) for k, v in zip(ingredients.ingredient_key, ingredients.rxcui) if pd.notna(v)}
DDINTER_ID = {k: str(v) for k, v in zip(ingredients.ingredient_key, ingredients.ddinter_id) if pd.notna(v)}

print(f"canonical ingredients: {len(ingredients)} | with an RxCUI: {len(RXCUI)} | with a DDInter id: {len(DDINTER_ID)}")
display(ingredients.head(3))

In [ ]:
# DDInter 2.0: severity per unordered pair of drug identifiers. One pair can be listed twice with
# different severities, so the higher one is kept rather than whichever row came last.
SEVERITY_RANK = {"Major": 3, "Moderate": 2, "Minor": 1}
interaction_rows = pd.read_csv(EXTERNAL_DIR / "ddinter" / "normalized" / "safety" / "drug_interactions.csv")
interactions = {}

for left, right, severity in zip(interaction_rows.ddinter_id_1, interaction_rows.ddinter_id_2,
                                 interaction_rows.severity):
    pair = frozenset((left, right))

    if SEVERITY_RANK.get(severity, 0) > SEVERITY_RANK.get(interactions.get(pair), 0):
        interactions[pair] = severity

print(f"DDInter pairs: {len(interactions)} | {Counter(interactions.values()).most_common()}")

**MED-RT alignment, done once.** MED-RT states a contraindication as (RxCUI, MeSH disease
descriptor), while the anamnesis linker of Section 5 produces UMLS CUIs. The MeSH-to-UMLS lookup was
run once against the UMLS source-identifier search and its outcome per descriptor is cached in
`data/external/umls_cache/medrt_disease_cuis.parquet`. A descriptor that returned several concepts
with no name agreement is left unmapped rather than guessed, which costs coverage and keeps the
alignment honest. The result is one flat table of `(rxcui, condition_cui)` pairs.

In [ ]:
medrt = pd.read_csv(EXTERNAL_DIR / "rx" / "rxclass" / "medrt_contraindications.csv")
ci_with = medrt[(medrt.rela == "ci_with") & (medrt.class_type == "DISEASE")].drop_duplicates(["rxcui", "class_id"])

mesh_to_cui = pd.read_parquet(UMLS_CACHE_DIR / "medrt_disease_cuis.parquet")
mesh_to_cui = mesh_to_cui[mesh_to_cui.umls_release == UMLS_VERSION]
print("MeSH descriptor to UMLS concept:")
display(mesh_to_cui.mapping_status.value_counts().rename("descriptors").to_frame())

resolved = mesh_to_cui[mesh_to_cui.mapping_status.isin(["mapped_unique", "mapped_by_name"])
                       & mesh_to_cui.disease_cui.notna()]
contraindications = (ci_with.merge(resolved[["class_id", "disease_cui", "umls_label"]], on="class_id")
                     .rename(columns={"disease_cui": "condition_cui", "umls_label": "condition_label"})
                     [["rxcui", "condition_cui", "condition_label"]])

CONTRAINDICATION_PAIRS = {(int(r.rxcui), r.condition_cui) for r in contraindications.itertuples()}
CONTRAINDICATION_LABEL = {(int(r.rxcui), r.condition_cui): r.condition_label for r in contraindications.itertuples()}

print(f"MED-RT ci_with pairs: {len(ci_with)} over {ci_with.class_id.nunique()} MeSH descriptors | "
      f"aligned to a UMLS concept: {len(contraindications)}")
display(contraindications.head(3))

### 3.3 Discharge therapy: template extraction

**Task.** Turn *Terapia alla Dimissione* into one row per prescription.
**Method.** Each prescription is a quoted entry with the shape
`Principio attivo (Prodotto forma dose): da assumere <posologia>`. Position declares the role of each
field, so a regex reads it; nothing has to be discovered.
**Output.** `prescriptions`, with `encOid`, `product_name`, `ingredients`, `dose_text`,
`instructions` and `atc_code`.
**Rationale.** This is a filled form, not prose. NER would have to learn a boundary the template
already states.

A combination product keeps its ingredient **list** and its dose **as written**:
`Clopidogrel/acido acetilsalicilico` with `dose_text = "75/100 mg"`. Splitting that into one number
would silently attribute the whole strength to one substance.

In [ ]:
# The template, in three regexes. `ENTRY` splits an entry at its first colon; `PRODUCT_TAIL` takes
# the LAST parenthetical of the head as the commercial product, which is what handles an entry whose
# ingredient field carries a parenthetical synonym of its own; `STRENGTH` reads a dose as written,
# combination strengths included.
ENTRY = re.compile(r"^(?P<head>[^:]+):(?P<body>.*)$", re.S)
PRODUCT_TAIL = re.compile(r"\(([^()]*)\)([^()]*)$")
FORM_LEAD = re.compile(r"\b(cp|cerott|soluz|sosp|comp|bust|gtt|fial|fl|capsul|ovul|crema|pomata|"
                       r"collirio|spray|scir|granul|polvere|nebuliz)", re.I)
STRENGTH = re.compile(r"\d+(?:[.,]\d+)?(?:\s*/\s*\d+(?:[.,]\d+)?)*\s*(?:mg/d|mg|mcg|g|UI|U|ml|%)", re.I)
NON_DRUG_THERAPY = re.compile(r"\b(ossigeno|niv|cpap|cannula nasale|maschera|ventil)", re.I)
NO_THERAPY = re.compile(r"nessun|non assume|nulla|assente", re.I)


def parse_prescription(entry):
    """One quoted therapy entry -> a prescription dict, or None when the entry is not one."""
    entry = entry.strip()

    if not entry or NON_DRUG_THERAPY.search(entry.split("(")[0]):   # oxygen and ventilation are not drugs
        return None

    match = ENTRY.match(entry)

    if not match:
        return None

    head, body = match.group("head"), match.group("body").strip()
    product = PRODUCT_TAIL.search(head)
    product_text = f"{product.group(1)} {product.group(2)}".strip() if product else None
    ingredient_field = re.sub(r"\([^()]*\)", " ", head[:product.start()] if product else head).strip()
    ingredient_list = [part.strip() for part in ingredient_field.split("/") if part.strip()]

    if not ingredient_list:
        return None

    product_name = None

    if product_text:
        form = FORM_LEAD.search(product_text)
        product_name = (product_text[:form.start()] if form else STRENGTH.sub("", product_text)).strip() or None

    instructions = re.sub(r"^\s*da assumere\s*", "", body).strip()
    dose = STRENGTH.search(instructions) or (STRENGTH.search(product_text) if product_text else None)

    return {"product_name": product_name, "ingredients": ingredient_list,
            "dose_text": dose.group(0) if dose else None, "instructions": instructions or None}


example = ('Clopidogrel/acido acetilsalicilico (Duoplavin cpr.riv. 75 mg): da assumere 75/100 mg (ore 19)')
print(example)
print(parse_prescription(example))

In [ ]:
# One pass over the cohort. Prescriptions are quoted; a report that quotes nothing and states an
# absence of therapy contributes no row.
rows = []

for patient in patients:
    text = report_text(patient, T_DISCHARGE)
    entries = re.findall(r'"([^"]*)"', text)

    if not entries:
        if not text.strip() or NO_THERAPY.search(text):
            continue

        entries = [text]

    for entry in entries:
        parsed = parse_prescription(entry)

        if parsed is not None:
            rows.append({"encOid": patient["encOid"], **parsed,
                         "atc_code": PRODUCT_ATC.get(normalize_name(parsed["product_name"] or ""))})

prescriptions = pd.DataFrame(rows)
prescriptions["prescription_id"] = prescriptions.index

print(f"prescriptions: {len(prescriptions)} over {prescriptions.encOid.nunique()} patients")
print(f"  combination products (more than one ingredient): {int((prescriptions.ingredients.map(len) > 1).sum())}")
print(f"  with an ATC code from the product registry: {prescriptions.atc_code.notna().sum()} "
      f"({prescriptions.atc_code.notna().mean():.0%})")
display(prescriptions[prescriptions.ingredients.map(len) > 1].head(4))

In [ ]:
# The ingredient view: one row per active ingredient of a prescription, carrying the identifiers the
# alert engine joins on. A combination product contributes one row per ingredient and keeps its
# prescription_id, which is what lets Section 6 tell a duplicate prescription from a combination.
prescribed = (prescriptions.explode("ingredients")
              .rename(columns={"ingredients": "ingredient"})
              .dropna(subset=["ingredient"])
              .reset_index(drop=True))
prescribed["ingredient_key"] = prescribed.ingredient.map(ingredient_key)
prescribed["preferred_label"] = prescribed.ingredient_key.map(PREFERRED_LABEL)
prescribed["rxcui"] = prescribed.ingredient_key.map(RXCUI).astype("Int64")
prescribed["ddinter_id"] = prescribed.ingredient_key.map(DDINTER_ID).astype("string")

prescriptions.to_parquet(OUTPUT_DIR / "discharge_prescriptions.parquet")

print(f"prescribed ingredient rows: {len(prescribed)} | distinct ingredients: {prescribed.ingredient_key.nunique()}")
display(pd.Series({
    "known to the AIFA ingredient crosswalk": prescribed.preferred_label.notna().mean(),
    "with an RxCUI (MED-RT drug side)": prescribed.rxcui.notna().mean(),
    "with a DDInter id (interaction rule)": prescribed.ddinter_id.notna().mean(),
}, name="share of ingredient rows").to_frame().style.format("{:.1%}"))

**Output of Section 3.** `prescriptions` holds one row per discharge prescription and `prescribed`
one row per active ingredient of one. Identifier coverage is a ceiling on the rules of Section 6: a
rule can only fire where the identifier it joins on exists, and the shares above say how often that
is. Prescriptions written outside the quoted block are not extracted, which Section 7 shows as
missing recall rather than as a hidden repair.

## 4. Admission therapy: few-shot extraction

**Task.** Find the medication mentions in *Terapia medica all'ingresso*, exactly as written.
**Method.** Few-shot generative information extraction with the local instruction model; a mention is
kept only if it can be located in the source text.
**Output.** `admission_mentions`, one row per grounded mention with its character offsets.
**Rationale.** Unlike the discharge report, the admission text has no stable template: it mixes
product lists with colons, bare lowercase lists, template-style entries naming active ingredients,
empty statements and non-drug interventions. A single generative extractor covers that variety with
one prompt.

The extractor is run on the 50 annotated patients only. Admission therapy feeds no downstream stage —
the alert engine of Section 6 reasons over the discharge regimen — so its only role is measurement,
and running a local model over all 857 reports would buy nothing.

```
admission text -> few-shot extraction -> drug mentions -> evaluation
```

In [ ]:
# One deterministic entry point to the local model, reused by this section and by Section 9.
local_llm = ollama.Client()


def generate_local(prompt, system=None, tools=None):
    """One call to the local instruction model. Returns the response message."""
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    response = local_llm.chat(model=LOCAL_LLM_MODEL, messages=messages, tools=tools,
                              options={"temperature": 0, "num_ctx": 4096})
    return response["message"]


print(f"local model: {LOCAL_LLM_MODEL} (temperature 0)")

**The prompt.** Four examples, all from **development** patients, chosen for the variants they show:
an empty therapy, manufacturer-suffixed product names, combination products, and an irregular entry
written without a colon. No development patient's admission text lists oxygen or ventilation, so that
case is stated in the instruction rather than demonstrated: borrowing a held-out report for it would
put test data in the prompt.

In [ ]:
FEW_SHOT_PATIENTS = [10149591, 10102082, 10083544, 10075109]
assert set(FEW_SHOT_PATIENTS) <= set(DEV_PATIENTS), "few-shot examples must come from development patients"

FEW_SHOT_ANSWERS = {
    10149591: [],
    10102082: ["Rivaroxaban doc", "Flecainide auro", "Lobivon"],
    10083544: ["Triatec hct", "Congescor", "Ezevast", "Pantoprazolo sand", "Furosemide l.f.m.", "Mirebax"],
    10075109: ["Eliquis", "Reaptan", "Sequacor", "Lanoxin", "Omeprazolo my", "enantone"],
}

EXTRACTION_INSTRUCTION = (
    "Estrai i farmaci della terapia domiciliare. "
    "Rispondi SOLO con una lista JSON di stringhe copiate esattamente dal testo. "
    "Non includere dosi, orari, forme farmaceutiche o istruzioni. "
    "Non includere ossigeno, NIV, CPAP o altri presidi non farmacologici. "
    "Se non ci sono farmaci, rispondi []."
)

FEW_SHOT_BLOCK = "\n\n".join(
    f"TESTO: {report_text(patient_by_oid[oid], T_ADMISSION)}\n"
    f"FARMACI: {json.dumps(FEW_SHOT_ANSWERS[oid], ensure_ascii=False)}"
    for oid in FEW_SHOT_PATIENTS)


def extraction_prompt(text):
    return f"{FEW_SHOT_BLOCK}\n\nTESTO: {text}\nFARMACI:"


print(EXTRACTION_INSTRUCTION)
print()
print(wrap(extraction_prompt("<testo del paziente>")))

In [ ]:
def parse_json_list(answer):
    """The JSON list inside a model answer, or an empty list when there is none."""
    start, end = answer.find("["), answer.rfind("]")

    if start < 0 or end < start:
        return []

    try:
        parsed = json.loads(answer[start:end + 1])
    except json.JSONDecodeError:
        return []

    return [x for x in parsed if isinstance(x, str)] if isinstance(parsed, list) else []


def ground_mentions(text, mentions):
    """Keep only mentions that occur in `text`, with their offsets.

    This is a grounding check, not a second extractor: it drops a medication name the model produced
    but the report does not contain, and gives the surviving ones the span an exact-span evaluation
    needs. Repeated mentions consume successive occurrences, so two predictions never claim one span.
    """
    lowered = text.lower()
    taken, grounded = [], []

    for mention in mentions:
        needle = mention.strip().lower()

        if not needle:
            continue

        start = lowered.find(needle)

        while start >= 0 and any(start < end and begin < start + len(needle) for begin, end in taken):
            start = lowered.find(needle, start + 1)

        if start < 0:
            continue

        taken.append((start, start + len(needle)))
        grounded.append({"mention": text[start:start + len(needle)], "start": start, "end": start + len(needle)})

    return grounded


def extract_admission_mentions(text):
    """Few-shot extraction followed by the grounding check."""
    if not text.strip():
        return []

    answer = generate_local(extraction_prompt(text), system=EXTRACTION_INSTRUCTION)["content"]
    return ground_mentions(text, parse_json_list(answer))

In [ ]:
# The generation is the expensive step, so it is cached. The sidecar records the model and the exact
# prompt, so changing either invalidates the cache instead of reusing answers from another prompt.
admission_cache = OUTPUT_DIR / "admission_mentions.parquet"
admission_meta = OUTPUT_DIR / "admission_mentions.meta.json"
admission_signature = {"model": LOCAL_LLM_MODEL, "instruction": EXTRACTION_INSTRUCTION,
                       "few_shot_patients": FEW_SHOT_PATIENTS}

if admission_cache.exists() and admission_meta.exists() \
        and json.loads(admission_meta.read_text(encoding="utf-8")) == admission_signature:
    admission_mentions = pd.read_parquet(admission_cache)
    print(f"loaded {len(admission_mentions)} cached admission mentions")
else:
    rows = []

    for oid in DEV_PATIENTS + TEST_PATIENTS:
        text = report_text(patient_by_oid[oid], T_ADMISSION)

        for mention in extract_admission_mentions(text):
            rows.append({"encOid": oid, **mention})

    admission_mentions = pd.DataFrame(rows, columns=["encOid", "mention", "start", "end"])
    admission_mentions.to_parquet(admission_cache)
    admission_meta.write_text(json.dumps(admission_signature), encoding="utf-8")
    print(f"extracted and cached {len(admission_mentions)} admission mentions")

print(f"patients with at least one mention: {admission_mentions.encOid.nunique()} of "
      f"{len(DEV_PATIENTS) + len(TEST_PATIENTS)}")

In [ ]:
# One worked example: the raw report, then what survived generation and grounding.
worked_oid = TEST_PATIENTS[0]
print(f"patient {worked_oid} | Terapia medica all'ingresso")
print(wrap(report_text(patient_by_oid[worked_oid], T_ADMISSION)))
display(admission_mentions[admission_mentions.encOid == worked_oid].reset_index(drop=True))

## 5. Anamnesis: NER, context and UMLS linking

**Task.** Turn free anamnesis prose into condition mentions a symbolic rule can consume.
**Method.** Three stages with three distinct failure modes.
**Output.** `conditions_linked`, one row per mention with its context axes and either a UMLS concept
or an explicit abstention.
**Rationale.** A span is not a fact and a string is not a concept: the mention has to be found, its
status decided, and its identity resolved, and each of those can fail on its own.

| stage | technique | decides |
|---|---|---|
| 5.1 mention detection | zero-shot NER (GLiNER) | *what* is mentioned |
| 5.2 context | ConText over an Italian cue list | *whether it holds*, and *for whom* |
| 5.4 linking | UMLS search, then contextual ranking | *which concept* |

### 5.1 Mention detection

In [ ]:
# The Italian model is used for sentence segmentation only: GLiNER runs per sentence and the offsets
# are mapped back to the document, so spans stay document-level.
nlp = spacy.load(SPACY_MODEL)

# Labels are entity TYPE NAMES, not descriptions: GLiNER matches a type against words, so a label
# that names two categories at once, or paraphrases one, is a different task.
NER_LABELS = ["malattia", "sintomo", "sostanza allergenica"]
CONDITION_LABELS = ["malattia", "sintomo"]     # a disease and a reported manifestation are both conditions

gliner_model = None


def get_gliner():
    """GLiNER, loaded on first use - a warm span cache means it is never loaded at all."""
    global gliner_model

    if gliner_model is None:
        gliner_model = GLiNER.from_pretrained(GLINER_MODEL).to(DEVICE)

    return gliner_model


def extract_entities(text):
    """Sentence-split `text`, tag each sentence, return spans with document-level offsets."""
    sentences = [(s.text, s.start_char) for s in nlp(text).sents if s.text.strip()]

    if not sentences:
        return []

    tagged = get_gliner().batch_predict_entities([s for s, _ in sentences], NER_LABELS,
                                                 threshold=NER_THRESHOLD)
    return [{"label": e["label"], "text": e["text"], "score": round(e["score"], 2),
             "start": offset + e["start"], "end": offset + e["end"], "sentence": sentence.strip()}
            for (sentence, offset), entities in zip(sentences, tagged) for e in entities]


print("labels:", NER_LABELS, "| threshold:", NER_THRESHOLD)

In [ ]:
# Tagging 857 reports is the expensive step, so it is cached with the settings that produced it.
entity_cache = OUTPUT_DIR / "anamnesis_entities.parquet"
entity_meta = OUTPUT_DIR / "anamnesis_entities.meta.json"
entity_signature = {"model": GLINER_MODEL, "labels": NER_LABELS, "threshold": NER_THRESHOLD}

if entity_cache.exists() and entity_meta.exists() \
        and json.loads(entity_meta.read_text(encoding="utf-8")) == entity_signature:
    entities = pd.read_parquet(entity_cache)
    print(f"loaded {len(entities)} cached anamnesis spans")
else:
    entities = pd.DataFrame([{"encOid": p["encOid"], **span}
                             for p in patients for span in extract_entities(report_text(p, T_ANAMNESIS))])
    entities.to_parquet(entity_cache)
    entity_meta.write_text(json.dumps(entity_signature), encoding="utf-8")
    print(f"tagged and cached {len(entities)} anamnesis spans")

display(entities.label.value_counts().rename("spans").to_frame())
print(f"mean spans per patient: {len(entities) / len(patients):.1f}")
display(entities[entities.encOid == patients[0]["encOid"]][["label", "text", "score", "sentence"]].head(8))

GLiNER tags a mention regardless of its context: `diabete mellito` is found inside `Nega diabete
mellito`, where the patient does not have it. Deciding that is stage 5.2.

### 5.2 Context: does the mention hold, and for whom?

Two **independent** axes, because one label cannot carry both: `Non familiarità per X` is negated
*and* about the family.

```
assertion:   affirmed | negated | possible
experiencer: patient  | family
```

ConText (Harkema et al., 2009) generalises NegEx: a cue is a typed modifier with a direction and a
scope, applied to every entity inside that scope. It is used here rather than hand-written scope code
because the scope semantics come from the published algorithm and the rules stay declarative data.
The scope runs to the sentence boundary and is cut early by a terminator, which is what handles the
coordinated lists this corpus is full of (`Nega A, B e C`). Cues are matched on the **surface form**: the
pipeline is `spacy.blank("it")` plus a sentencizer, so no lemma or part-of-speech information exists
in it, and none is needed for a cue list this small and this closed.

In [ ]:
# Italian cue vocabularies, curated on this corpus. Multi-word cues stay phrases because the single
# word means something else: "negativo" reports a result backwards, "negativo per X" negates X forwards.
NEGATION_FORWARD = {"nega", "negato", "negata", "non", "senza", "nessun", "nessuno", "mai", "assenza di",
                    "negativo per", "negativa per", "negativi per", "negative per",
                    "asintomatico per", "asintomatica per"}
NEGATION_BACKWARD = {"negativo", "negativa", "negato", "negata", "assente", "escluso", "esclusa"}
UNCERTAINTY = {"sospetto", "sospetta", "sospetti", "sospette", "possibile", "possibili", "probabile",
               "probabili", "verosimile", "verosimilmente", "dubbio", "dubbia", "presunto", "presunta"}
FAMILY = {"familiarita", "familiarità", "familiare", "madre", "padre", "fratello", "sorella", "figlio",
          "figlia", "zio", "zia", "nonno", "nonna", "genitore", "materno", "paterno"}
# Section headers close an open scope: "Allergie e intolleranze non note Anamnesi Remota: Ipertensione"
# must not negate what follows the header.
SECTION_HEADERS = {"anamnesi remota", "anamnesi patologica remota", "anamnesi prossima",
                   "anamnesi patologica prossima", "anamnesi familiare", "anamnesi fisiologica",
                   "fattori di rischio", "allergie e intolleranze", "familiarita", "familiarità"}
TERMINATORS = {"ma", "pero", "però", "tuttavia", "mentre", "riferisce", "presenta", "lamenta"} | SECTION_HEADERS


def build_context_pipeline():
    """Parser-free ConText pipeline: blank Italian tokenizer, sentence boundaries, cue rules."""
    rules = [ConTextRule(cue, "NEGATED_EXISTENCE", direction="FORWARD") for cue in sorted(NEGATION_FORWARD)]
    rules += [ConTextRule(cue, "NEGATED_EXISTENCE", direction="BACKWARD") for cue in sorted(NEGATION_BACKWARD)]
    rules += [ConTextRule(cue, "POSSIBLE_EXISTENCE", direction="FORWARD") for cue in sorted(UNCERTAINTY)]
    rules += [ConTextRule(cue, "FAMILY", direction="FORWARD") for cue in sorted(FAMILY)]
    rules += [ConTextRule(cue, "TERMINATE", direction="TERMINATE") for cue in sorted(TERMINATORS)]

    pipeline = spacy.blank("it")
    pipeline.add_pipe("sentencizer")
    pipeline.add_pipe("medspacy_context", config={"rules": None, "max_scope": None})  # rules=None: no English defaults
    pipeline.get_pipe("medspacy_context").add(rules)
    return pipeline


context_nlp = build_context_pipeline()
CONTEXT_DEFAULT = {"assertion": "affirmed", "experiencer": "patient"}


def context_axes(text, spans, pipeline=None):
    """One axis dict per span in `spans` = [(start, end), ...]."""
    pipeline = pipeline or context_nlp
    doc = pipeline.make_doc(text)
    kept, taken = [], set()

    for index, (start, end) in enumerate(spans):
        span = doc.char_span(int(start), int(end), label="CONDITION", alignment_mode="expand")

        if span is None or set(range(span.start, span.end)) & taken:   # spaCy forbids overlapping ents
            continue

        taken |= set(range(span.start, span.end))
        kept.append((index, span))

    doc.ents = [span for _, span in kept]

    for _, component in pipeline.pipeline:
        doc = component(doc)

    axes = [dict(CONTEXT_DEFAULT, cues="") for _ in spans]
    entity_at = {entity.start: entity for entity in doc.ents}

    for index, span in kept:
        cues = {modifier.category for modifier in entity_at[span.start]._.modifiers}
        # An explicit uncertainty cue outranks negation: "non escluse possibili stenosi" is uncertain,
        # not absent. Fixed precedence, so the result does not depend on modifier order.
        axes[index] = {"assertion": "possible" if "POSSIBLE_EXISTENCE" in cues
                       else "negated" if "NEGATED_EXISTENCE" in cues else "affirmed",
                       "experiencer": "family" if "FAMILY" in cues else "patient",
                       "cues": "; ".join(sorted(cues))}

    return axes


print(f"ConText pipeline: {context_nlp.pipe_names}")

In [ ]:
# Which modifier reaches which mention, on three real sentence shapes.
examples = [("Nega diabete mellito e dislipidemia.", ["diabete mellito", "dislipidemia"]),
            ("Non familiarita per malattie cardiovascolari.", ["malattie cardiovascolari"]),
            ("Sospetta amiloidosi cardiaca.", ["amiloidosi cardiaca"])]

display(pd.DataFrame([{"sentence": sentence, "mention": mention, "modifiers": axes["cues"] or "(none)",
                       "assertion": axes["assertion"], "experiencer": axes["experiencer"]}
                      for sentence, mentions in examples
                      for mention, axes in zip(mentions, context_axes(
                          sentence, [(sentence.find(m), sentence.find(m) + len(m)) for m in mentions]))]))

Both coordinated items after `Nega` are negated: the scope runs to the sentence boundary rather than
stopping at the first mention. In the second sentence the negation and the family cue land on the
same mention and write to different axes, which is why the axes are kept independent.

In [ ]:
# Apply the context model to every condition mention in the cohort.
conditions = entities[entities.label.isin(CONDITION_LABELS)].copy()
axes_rows = []

for oid, group in conditions.groupby("encOid"):
    text = report_text(patient_by_oid[oid], T_ANAMNESIS)
    axes_rows.append(pd.DataFrame(context_axes(text, list(zip(group.start, group.end))), index=group.index))

conditions = conditions.join(pd.concat(axes_rows))
conditions["usable"] = (conditions.assertion == "affirmed") & (conditions.experiencer == "patient")

print(f"condition mentions: {len(conditions)} over {conditions.encOid.nunique()} patients")
display(pd.DataFrame({"assertion": conditions.assertion.value_counts(),
                      "experiencer": conditions.experiencer.value_counts()}).fillna(0).astype(int))
print(f"usable (affirmed and about the patient): {int(conditions.usable.sum())} "
      f"({conditions.usable.mean():.0%})")
display(conditions.loc[~conditions.usable, ["text", "cues", "assertion", "experiencer", "sentence"]].head(6))

A condition is **usable** by a downstream rule only when it is affirmed and about the patient:

```python
usable = (assertion == "affirmed") and (experiencer == "patient")
```

Nothing else gates a rule. Whether a condition is current or historical is not modelled: no rule in
this pipeline reads it, and the cue vocabularies above carry no temporal category.

### 5.3 Syntactic scope: a dependency-parsing check

Scope could in principle be read off the syntax instead of the sentence: a cue would modify a mention
only when it governs it in the dependency tree. Two real sentences from the corpus are parsed below to
see whether the general-purpose Italian parser supports that on this register.

In [ ]:
# Two sentences that occur verbatim in the corpus; the gold annotates every condition in both as negated.
DEPENDENCY_EXAMPLES = [("Nega diabete mellito e dislipidemia.", "nega", ["diabete mellito", "dislipidemia"]),
                       ("Non familiarita per MCI e cardiopatie congenite.", "non", ["MCI", "cardiopatie congenite"])]

rows = []

for sentence, cue_word, mentions in DEPENDENCY_EXAMPLES:
    parsed = nlp(sentence)
    cue = next(token for token in parsed if token.text.lower() == cue_word)
    governed = set(cue.subtree)

    for mention in mentions:
        start = sentence.find(mention)
        head = parsed.char_span(start, start + len(mention), alignment_mode="expand").root
        rows.append({"sentence": sentence, "cue": cue.text, "cue POS": cue.pos_, "cue relation": cue.dep_,
                     "mention": mention, "mention head": head.text,
                     "cue governs the mention": head in governed})

display(pd.DataFrame(rows))
print("tokens governed by each cue:")

for sentence, cue_word, _ in DEPENDENCY_EXAMPLES:
    parsed = nlp(sentence)
    cue = next(token for token in parsed if token.text.lower() == cue_word)
    print(f"  {cue.text!r} in {sentence!r}: "
          f"{[t.text for t in cue.subtree if t is not cue] or '(nothing - the subtree is the cue alone)'}")

displacy.render(nlp(DEPENDENCY_EXAMPLES[0][0]), style="dep", jupyter=True,
                options={"compact": True, "distance": 95, "word_spacing": 20})

**Conclusion.** A scope rule built on the parse would keep exactly the pairs the table marks as
governed and drop the rest — and where it drops one, it drops a negation the sentence states plainly.
Both sentences coordinate two conditions under one cue, which is the construction that decides whether
such a rule is usable on this register. Dependency structure was therefore explored and not adopted
operationally: ConText's sentence scope needs no parser, so it is not exposed to the parser's analysis
of telegraphic clinical Italian. This is a qualitative observation on two sentences, not a measured
comparison — no F1 is computed for a dependency-based variant.

### 5.4 Linking the conditions to UMLS

**Task.** Map each condition mention to a UMLS Concept Unique Identifier, or abstain.
**Method.** Lexical candidate generation through the UTS search endpoint, then one semantic ranking
signal; a floor and a margin decide whether to commit.
**Output.** `conditions_linked`, one row per mention with a CUI or a named abstention.
**Rationale.** A rule needs a concept, not a string. UMLS supplies concept identity only — it asserts
nothing clinical; it is what lets an Italian mention and an English MeSH disease class refer to the
same thing in Section 6.

```
mention -> candidate generation -> candidate disambiguation -> CUI or NIL
```

| exact search returns | what happens | status |
|---|---|---|
| exactly one concept | accepted | `linked_exact` |
| several concepts | the sentence chooses between them | `linked_contextual` |
| nothing | the looser word search is asked, then ranked | `linked_contextual` |
| nothing at all | abstain | `nil_no_candidates` |

Ranking compares the mention **inside its own sentence** with each candidate's label and semantic
types, in a multilingual embedding space, because the mention is Italian and the label is usually
English. Two further abstentions come from the ranking itself: `nil_low_score` below the floor and
`nil_low_margin` when the top two candidates are effectively tied.

In [ ]:
UMLS_MIN_INTERVAL = 0.25       # at most four requests per second, as the UTS terms ask


def umls_search(term, search_type):
    """Concepts whose atoms match `term`, restricted server-side to the Disorders semantic group."""
    time.sleep(UMLS_MIN_INTERVAL)
    response = requests.get(f"{UMLS_BASE_URL}/search/{UMLS_VERSION}", timeout=30,
                            params={"string": term, "searchType": search_type, "returnIdType": "concept",
                                    "semanticGroups": UMLS_SEMANTIC_GROUP, "pageSize": 50,
                                    "apiKey": UMLS_API_KEY})
    response.raise_for_status()

    return [{"cui": r["ui"], "label": r.get("name", ""), "semantic_types": tuple(r.get("semanticTypes") or ())}
            for r in response.json().get("result", {}).get("results", [])
            if r.get("ui") and r["ui"] != "NONE"]     # some releases signal "no result" with a NONE row

In [ ]:
# The API answer depends on the string alone, so it is cached by surface form and release. A query
# that returned nothing is a recorded answer too, otherwise every run asks it again.
def load_umls_cache():
    """(candidates by (query, search type), queries already answered) from the shipped cache."""
    cached, answered = defaultdict(list), set()
    candidate_file = UMLS_CACHE_DIR / "search_candidates.parquet"
    status_file = UMLS_CACHE_DIR / "search_status.parquet"

    if candidate_file.exists():
        table = pd.read_parquet(candidate_file)
        table = table[table.umls_release == UMLS_VERSION].sort_values("candidate_rank")

        for row in table.itertuples():
            types = () if row.semantic_types is None else tuple(row.semantic_types)
            cached[(row.normalised_query, row.search_type)].append(
                {"cui": row.cui, "label": row.candidate_label, "semantic_types": types})

    if status_file.exists():
        table = pd.read_parquet(status_file)
        answered = {(r.normalised_query, r.search_type)
                    for r in table[table.umls_release == UMLS_VERSION].itertuples()}

    return cached, answered


umls_cached, umls_answered = load_umls_cache()
umls_new_queries = []


def umls_candidates(mention):
    """(search type, candidates) for one surface form: exact first, the looser word search only if
    exact returned nothing. Returns an empty list when neither the cache nor the API can answer."""
    query = normalize_name(mention)

    for search_type in ("exact", "words"):
        key = (query, search_type)

        if key not in umls_answered:
            if not UMLS_API_KEY:
                return search_type, []

            umls_cached[key] = umls_search(query, search_type)
            umls_answered.add(key)
            umls_new_queries.append(key)

        if umls_cached[key]:
            return search_type, umls_cached[key][:UMLS_MAX_CANDIDATES]

    return "words", []


def save_umls_cache():
    """Append the queries answered in this run to the on-disk cache."""
    if not umls_new_queries:
        return

    rows = [{"query": query, "normalised_query": query, "search_type": search_type,
             "candidate_rank": rank, "cui": candidate["cui"], "candidate_label": candidate["label"],
             "semantic_types": list(candidate["semantic_types"]), "umls_release": UMLS_VERSION}
            for query, search_type in umls_new_queries
            for rank, candidate in enumerate(umls_cached[(query, search_type)])]
    status = [{"normalised_query": query, "search_type": search_type, "umls_release": UMLS_VERSION}
              for query, search_type in umls_new_queries]

    for path, new_rows in ((UMLS_CACHE_DIR / "search_candidates.parquet", rows),
                           (UMLS_CACHE_DIR / "search_status.parquet", status)):
        if not new_rows:
            continue

        existing = pd.read_parquet(path) if path.exists() else pd.DataFrame()
        pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True).to_parquet(path, index=False)

    umls_new_queries.clear()


print(f"UMLS cache: {sum(len(v) for v in umls_cached.values())} candidates over "
      f"{len(umls_answered)} answered queries")

In [ ]:
embedder = SentenceTransformer(EMBEDDER_MODEL, device=DEVICE)


def rank_candidates(mention, sentence, candidates):
    """Candidates ordered by cosine similarity to the mention in its own sentence.

    Only the containing sentence is used, never the whole anamnesis: the patient's other conditions
    would drown the local context that actually disambiguates. The query:/passage: prefixes are the
    ones multilingual E5 was trained with.
    """
    texts = [f"query: {mention}. {sentence}".strip()] + \
            [f"passage: {c['label']} [{', '.join(c['semantic_types'])}]".strip() for c in candidates]
    vectors = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    scored = [{**candidate, "cosine": round(float(vectors[0] @ vector), 4)}
              for candidate, vector in zip(candidates, vectors[1:])]

    return sorted(scored, key=lambda c: -c["cosine"])


def abstain(status, candidate_count):
    return {"umls_cui": None, "umls_label": None, "umls_semantic_types": None,
            "umls_score": None, "score_margin": None, "link_status": status,
            "candidate_count": candidate_count}


def accept(candidate, status, margin):
    return {"umls_cui": candidate["cui"], "umls_label": candidate["label"],
            "umls_semantic_types": "; ".join(candidate["semantic_types"]),
            "umls_score": candidate.get("cosine"), "score_margin": margin,
            "link_status": status, "candidate_count": candidate.get("candidate_count")}


def link_condition(mention, sentence, found=None):
    """Link one Italian condition mention to a UMLS concept, or abstain with a named reason."""
    search_type, candidates = found if found is not None else umls_candidates(mention)

    if not candidates:
        return abstain("nil_no_candidates", 0)

    if search_type == "exact" and len(candidates) == 1:
        # An exact search says the string matched an atom of that concept - not that the concept is
        # the one the clinician meant. Section 7 measures how often accepting it is acceptable.
        return accept({**candidates[0], "candidate_count": 1}, "linked_exact", None)

    ranked = rank_candidates(mention, sentence, candidates)
    best = ranked[0]
    margin = round(best["cosine"] - ranked[1]["cosine"], 4) if len(ranked) > 1 else None

    if best["cosine"] < UMLS_COSINE_FLOOR:
        return abstain("nil_low_score", len(ranked))

    if margin is not None and margin < UMLS_COSINE_MARGIN:
        return abstain("nil_low_margin", len(ranked))

    return accept({**best, "candidate_count": len(ranked)}, "linked_contextual", margin)

In [ ]:
# Candidate generation is asked once per distinct surface form; the decision is taken once per
# mention, because the sentence is what settles which candidate is meant.
surface_forms = sorted(conditions.text.astype(str).str.strip().unique())
candidates_by_form = {form: umls_candidates(form) for form in surface_forms}
save_umls_cache()

decisions = [link_condition(str(row.text).strip(), str(row.sentence),
                            candidates_by_form[str(row.text).strip()])
             for row in conditions.itertuples()]

conditions_linked = pd.concat([conditions.reset_index(drop=True),
                               pd.DataFrame(decisions)], axis=1)
conditions_linked.to_parquet(OUTPUT_DIR / "conditions_linked.parquet")

print(f"condition mentions: {len(conditions_linked)} | distinct surface forms: {len(surface_forms)}")
display(conditions_linked.link_status.value_counts().rename("mentions").to_frame())
display(conditions_linked[conditions_linked.link_status.isin(UMLS_LINKED)]
        [["text", "umls_cui", "umls_label", "umls_score", "link_status"]]
        .drop_duplicates("text").head(8).reset_index(drop=True))

> **Experimental output, not suitable for clinical use.** These links are the output of a linker
> measured on a 120-mention single-annotator sample of the held-out patients (Section 7.5). That says
> how often it returns an acceptable concept on mentions of that kind; it establishes nothing about
> clinical safety.

**Output of Section 5.** `conditions_linked` carries, per mention, its two context axes and either a
UMLS concept or a named abstention. Together with `prescribed` it is the semantic backbone the alert
engine and the knowledge graph consume.

## 6. Symbolic alert engine

**Task.** Derive patient-level medication-safety alerts from the extracted facts.
**Method.** Four deterministic rules, one function each, over identifiers rather than strings.
**Output.** `alerts`, one row per firing with the identifiers behind it.
**Rationale.** No model decides whether an interaction or a contraindication exists. Each rule is a
set membership or a table lookup against an external knowledge base, so every alert can be
reconstructed rather than believed.

**Scope.** The rules run on the **discharge regimen** — what the patient takes after the stay. The
admission list feeds no rule.

| Rule | Fires when | Joins on | Knowledge base |
|---|---|---|---|
| duplicate ingredient | two prescriptions share an ingredient | `ingredient_key` | none: derived from the prescriptions |
| drug-drug interaction | two prescribed ingredients interact | `ingredient_key` → `ddinter_id` | DDInter 2.0 |
| recorded allergy / intolerance | a prescribed ingredient is a recorded allergen | `ingredient_key` | the patient's own anamnesis |
| drug-condition contraindication | a prescribed drug is contraindicated in an affirmed condition | `rxcui` and `umls_cui` | MED-RT, aligned in Section 3 |

### The alerts are derived facts

None of these is written anywhere in the record. Each is inferred from facts that are:

```
patient takes drug A                          patient takes a drug with RxCUI X
patient takes drug B                          patient has an affirmed condition with CUI Y
DDInter: A interacts with B                   MED-RT: (X, Y) is a contraindication
------------------------------                ------------------------------------------
patient has a drug-drug interaction alert     patient has a drug-condition alert
```

The premises come from three different places — two reports and an external table — and the
conclusion from none of them. That is what the symbolic layer contributes; the chain is also where it
is fragile, since every premise is itself an extraction or a linking decision.

Allergy reasoning is **exact ingredient identity only**. ATC classes are not used for it: ATC is a
therapeutic classification, not a cross-reactivity ontology, and a recorded allergy to a class that
cannot be mapped to an ingredient is left unresolved rather than turned into an alert.

### 6.1 Recorded allergies and intolerances

In [ ]:
# The allergy field has two registers. The template writes an explicit parenthesised list
# ("Allergie: Principi attivi (Cefazolina)"); free prose adds the rest, which GLiNER's
# "sostanza allergenica" label picks up. Both feed one candidate set.
SECTION_RE = re.compile(r"(anamnesi\s+(?:patologica\s+)?remota|anamnesi\s+(?:patologica\s+)?prossima|"
                        r"anamnesi\s+familiare|anamnesi\s+fisiologica|fattori\s+di\s+rischio|"
                        r"allergie\s+e\s+intolleranze|familiarit[aà])\s*:", re.I)
ALLERGY_LIST_RE = re.compile(r"\b(principi\s+attivi|alimenti|farmaci|altro|intolleranz\w*|allergie)\s*\(", re.I)
ALLERGY_CUE_RE = re.compile(r"allerg|intolleran", re.I)


def allergy_section(text):
    """(start, end) of the 'Allergie e intolleranze' section, or None."""
    headers = sorted((m.start(), m.group(1).lower()) for m in SECTION_RE.finditer(text))

    for index, (start, name) in enumerate(headers):
        if name.startswith("allergie"):
            return start, (headers[index + 1][0] if index + 1 < len(headers) else len(text))

    return None


def matching_paren(text, opening):
    """Index of the parenthesis closing the one at `opening`; the lists nest a reason inside."""
    depth = 0

    for index in range(opening, len(text)):
        depth += (text[index] == "(") - (text[index] == ")")

        if depth == 0:
            return index

    return -1


def template_reactions(text):
    """(reaction type, substance, start, end) for each substance listed in the allergy template."""
    section = allergy_section(text)

    if section is None:
        return []

    offset, end = section
    segment = text[offset:end]
    found = []

    for header in ALLERGY_LIST_RE.finditer(segment):
        reaction_type = "intolerance" if header.group(1).lower().startswith("intolleranz") else "allergy"
        opening = segment.index("(", header.start())
        closing = matching_paren(segment, opening)

        if closing < 0:
            continue

        inner = segment[opening + 1:closing]
        # blank out a nested reason "(capogiri)" while preserving the offsets of what remains
        masked = re.sub(r"\([^)]*\)", lambda m: " " * len(m.group(0)), inner)

        for item in re.finditer(r"[^,;.]+", masked):
            substance = item.group(0).strip()

            if 2 < len(substance) < 60:
                start = offset + opening + 1 + item.start() + (len(item.group(0)) - len(item.group(0).lstrip()))
                found.append((reaction_type, substance, start, start + len(substance)))

    return found

In [ ]:
# A GLiNER allergen span is kept only in an allergy context: the label tags a substance wherever it
# appears, so "Sostituita flecainide con nadololo" is a therapy change, not an allergy. Flagging a
# patient as allergic to a drug they currently take is the worst error a safety rule can make.
def cue_introduces(sentence, mention):
    """True when an allergy cue precedes the mention in its sentence, i.e. introduces the list."""
    position = str(sentence).lower().find(str(mention).lower())
    cue = ALLERGY_CUE_RE.search(str(sentence))

    return cue is not None and (position < 0 or cue.start() < position)


reaction_rows = []

for patient in patients:
    text = report_text(patient, T_ANAMNESIS)

    for reaction_type, substance, start, end in template_reactions(text):
        reaction_rows.append({"encOid": patient["encOid"], "source": "template", "reaction_type": reaction_type,
                              "mention": substance, "start": start, "end": end})

allergen_spans = entities[entities.label == "sostanza allergenica"]

for oid, group in allergen_spans.groupby("encOid"):
    text = report_text(patient_by_oid[oid], T_ANAMNESIS)
    section = allergy_section(text)

    for span in group.itertuples():
        in_section = section is not None and section[0] <= span.start < section[1]

        if in_section or cue_introduces(span.sentence, span.text):
            reaction_rows.append({"encOid": oid, "source": "GLiNER", "reaction_type": "allergy",
                                  "mention": span.text, "start": int(span.start), "end": int(span.end)})

reactions = pd.DataFrame(reaction_rows)
print(f"allergen spans from GLiNER: {len(allergen_spans)} | kept in an allergy context: "
      f"{int((reactions.source == 'GLiNER').sum())}")
print(f"recorded reactions: {len(reactions)} | {reactions.source.value_counts().to_dict()} | "
      f"{reactions.reaction_type.value_counts().to_dict()}")

In [ ]:
# "Nega allergie note" must fire nothing, so the same context model that gates the conditions gates
# these mentions too, rather than an ad-hoc check.
context_by_index = {}

for oid, group in reactions.groupby("encOid"):
    text = report_text(patient_by_oid[oid], T_ANAMNESIS)

    for index, axes in zip(group.index, context_axes(text, list(zip(group.start, group.end)))):
        context_by_index[index] = axes["assertion"] == "affirmed" and axes["experiencer"] == "patient"

asserted = pd.Series(context_by_index)
print(f"denied or non-patient mentions dropped: {int((~asserted).sum())} of {len(reactions)}")
reactions = reactions[asserted].copy()


def allergen_ingredients(mention):
    """Canonical ingredients an allergen mention denotes, by exact identity only.

    Two exact routes: the substance is itself a catalogued ingredient, or it is a registered product
    that contains exactly one. A product with several ingredients is not expanded - an allergy to a
    combination does not implicate each component - and a class term such as "cefalosporine" stays
    unresolved rather than becoming an alert.
    """
    keys = {key for key in (ingredient_key(part) for part in re.split(r"[/+]", str(mention)))
            if key in KNOWN_INGREDIENTS}

    if keys:
        return sorted(keys)

    product_keys = {ingredient_key(name) for name in PRODUCT_INGREDIENTS.get(normalize_name(mention), [])}

    return sorted(product_keys) if len(product_keys) == 1 and product_keys <= KNOWN_INGREDIENTS else []


reactions["ingredient_keys"] = reactions.mention.map(allergen_ingredients)
resolved_reactions = reactions[reactions.ingredient_keys.map(len) > 0]
print(f"resolved to a canonical ingredient: {len(resolved_reactions)} of {len(reactions)}")
display(resolved_reactions[["encOid", "source", "reaction_type", "mention", "ingredient_keys"]].head(8))
print("left unresolved (class terms, foods, environmental allergens):",
      sorted(set(reactions.loc[reactions.ingredient_keys.map(len) == 0, "mention"].str.lower()))[:10])

### 6.2 The four rules

In [ ]:
ALERT_COLUMNS = ["encOid", "alert_type", "drug_a", "drug_b", "condition", "allergen",
                 "severity", "source", "detail", "drug_a_key", "drug_b_key", "drug_rxcui",
                 "condition_cui", "mention_id"]


def find_duplicate_ingredients(prescribed):
    """Rule 1 - the same canonical ingredient in two or more prescriptions of one patient.

    Grouped by prescription and not by row, so a combination product (two ingredients, one
    prescription) is not a duplication and neither are two ingredient rows of one entry.
    """
    alerts = []

    for oid, rows in prescribed.groupby("encOid"):
        for key, group in rows.groupby("ingredient_key"):
            if group.prescription_id.nunique() < 2:
                continue

            products = sorted({str(name) for name in group.product_name.dropna()})
            alerts.append({"encOid": oid, "alert_type": "duplicate_ingredient",
                           "drug_a": PREFERRED_LABEL.get(key, key), "drug_a_key": key,
                           "severity": "duplication", "source": "discharge prescriptions",
                           "detail": f"{key} in {group.prescription_id.nunique()} prescriptions "
                                     f"({', '.join(products) or 'no product named'})"})

    return alerts


def find_ddi_alerts(prescribed):
    """Rule 2 - a DDInter interaction between two ingredients prescribed separately.

    Two components of one fixed-dose product are not a co-prescription of two drugs, so pairs that
    occur inside a single prescription are excluded.
    """
    alerts = []

    for oid, rows in prescribed.groupby("encOid"):
        within_product = {frozenset(pair)
                          for _, group in rows.groupby("prescription_id")
                          for pair in combinations(sorted(set(group.ingredient_key)), 2)}
        identified = sorted({(str(r.ddinter_id), r.ingredient_key)
                             for r in rows.itertuples() if pd.notna(r.ddinter_id)})

        for (id_a, key_a), (id_b, key_b) in combinations(identified, 2):
            severity = interactions.get(frozenset((id_a, id_b)))

            if severity not in ("Major", "Moderate") or frozenset((key_a, key_b)) in within_product:
                continue

            alerts.append({"encOid": oid, "alert_type": "drug_drug_interaction",
                           "drug_a": PREFERRED_LABEL.get(key_a, key_a), "drug_a_key": key_a,
                           "drug_b": PREFERRED_LABEL.get(key_b, key_b), "drug_b_key": key_b,
                           "severity": severity, "source": "DDInter 2.0",
                           "detail": f"{key_a} + {key_b}"})

    return alerts


def find_drug_reaction_alerts(prescribed, reactions):
    """Rule 3 - a prescribed ingredient is a recorded allergen or intolerance of that patient.

    Exact canonical identity, and one alert per (ingredient, reaction type): the template list and the
    GLiNER spans routinely name the same substance twice.
    """
    alerts = []
    by_patient = dict(list(reactions.groupby("encOid"))) if len(reactions) else {}

    for oid, rows in prescribed.groupby("encOid"):
        recorded = by_patient.get(oid)

        if recorded is None:
            continue

        prescribed_keys = set(rows.ingredient_key)
        fired = set()

        for record in recorded.itertuples():
            for key in record.ingredient_keys:
                if key not in prescribed_keys or (key, record.reaction_type) in fired:
                    continue

                fired.add((key, record.reaction_type))
                alerts.append({"encOid": oid, "alert_type": f"drug_{record.reaction_type}",
                               "drug_a": PREFERRED_LABEL.get(key, key), "drug_a_key": key,
                               "allergen": record.mention, "severity": record.reaction_type,
                               "source": f"anamnesis ({record.source})",
                               "detail": f"{key} prescribed; {record.reaction_type} to {record.mention}"})

    return alerts


def find_drug_condition_alerts(prescribed, conditions_linked):
    """Rule 4 - a prescribed drug is contraindicated in a condition the patient is asserted to have.

    Both sides must carry an identifier: the drug an RxCUI, the condition a UMLS CUI reached by the
    linker on a usable mention. Matching is exact CUI equality, so no hierarchy is traversed and a
    condition more specific than the MED-RT class does not match.
    """
    alerts = []
    eligible = conditions_linked[conditions_linked.usable
                                 & conditions_linked.link_status.isin(UMLS_LINKED)
                                 & conditions_linked.umls_cui.notna()]

    for oid, rows in prescribed.groupby("encOid"):
        patient_conditions = eligible[eligible.encOid == oid]
        fired = set()

        for drug in rows[rows.rxcui.notna()].drop_duplicates("ingredient_key").itertuples():
            for mention in patient_conditions.itertuples():
                pair = (int(drug.rxcui), mention.umls_cui)

                if pair not in CONTRAINDICATION_PAIRS or pair in fired:
                    continue

                fired.add(pair)
                alerts.append({"encOid": oid, "alert_type": "drug_condition_contraindication",
                               "drug_a": PREFERRED_LABEL.get(drug.ingredient_key, drug.ingredient_key),
                               "drug_a_key": drug.ingredient_key, "drug_rxcui": int(drug.rxcui),
                               "condition": mention.text, "condition_cui": mention.umls_cui,
                               "mention_id": f"{oid}_{int(mention.start)}_{int(mention.end)}",
                               "severity": "contraindication", "source": "MED-RT",
                               "detail": f"{drug.ingredient_key} contraindicated in "
                                         f"{CONTRAINDICATION_LABEL[pair]} (mention: {mention.text})"})

    return alerts


def run_alert_rules(prescribed, conditions_linked, reactions):
    """Every rule over one set of inputs. Section 7 calls this again with annotated inputs."""
    return pd.DataFrame(find_duplicate_ingredients(prescribed)
                        + find_ddi_alerts(prescribed)
                        + find_drug_reaction_alerts(prescribed, reactions)
                        + find_drug_condition_alerts(prescribed, conditions_linked),
                        columns=ALERT_COLUMNS)

In [ ]:
alerts = run_alert_rules(prescribed, conditions_linked, resolved_reactions)
alerts.to_parquet(OUTPUT_DIR / "alerts.parquet")

display(alerts.alert_type.value_counts().rename("alerts").to_frame())
print(f"patients with at least one alert: {alerts.encOid.nunique()} of {len(patients)}")
print("interaction severity:",
      alerts[alerts.alert_type == "drug_drug_interaction"].severity.value_counts().to_dict())

In [ ]:
# One patient's alerts, with the identifiers each rule fired on. The patient is chosen so that
# several rules are visible at once: among those carrying the most distinct alert types, the one with
# the fewest alerts, so the table stays readable.
variety = alerts.groupby("encOid").agg(types=("alert_type", "nunique"), total=("alert_type", "size"))
example_patient = variety[variety.types == variety.types.max()].total.idxmin()
print(f"every alert of patient {example_patient}:")
display(alerts[alerts.encOid == example_patient]
        [["alert_type", "drug_a", "drug_b", "condition", "allergen", "severity", "source", "detail"]]
        .reset_index(drop=True))

**Output of Section 6.** Each rule inherits the reliability of the facts it runs on. The duplicate
and interaction rules join on identifiers that the discharge template states almost verbatim; the
contraindication rule sits at the end of a two-step chain — an Italian phrase to a UMLS concept, and
a MeSH descriptor to the same vocabulary — so it is the one most exposed to upstream error. Section 7
measures that difference rather than assuming it.

## 7. Evaluation

**Task.** Say how well each stage works, on data no decision was made against.
**Method.** One metric per question, computed on the **30 held-out patients** only.
**Output.** A scorecard, with a few representative errors under each stage that has them.
**Rationale.** These tasks are not commensurable, so nothing is averaged into a single score. Every
number below comes from patients that were not looked at while building the pipeline.

In [ ]:
GOLD = TEST_PATIENTS

gold_discharge = pd.read_csv(ANNOTATION_DIR / "gold_discharge_medications_all.csv")
gold_discharge["active_ingredients"] = gold_discharge.active_ingredients.map(json.loads)
gold_admission = pd.read_csv(ANNOTATION_DIR / "gold_ingress_medications.csv")
gold_context = pd.read_csv(ANNOTATION_DIR / "gold_anamnesis_context.csv")
gold_links = pd.read_csv(ANNOTATION_DIR / "gold_umls_links.csv")
gold_links["acceptable_cuis"] = gold_links.acceptable_cuis.map(json.loads)
gold_reactions = pd.read_csv(ANNOTATION_DIR / "gold_allergens.csv")

# Restricted to the held-out patients once, here, so no cell below can report on a development patient.
gold_discharge = gold_discharge[gold_discharge.encOid.isin(GOLD)]
gold_admission = gold_admission[gold_admission.encOid.isin(GOLD)]
gold_context = gold_context[gold_context.encOid.isin(GOLD)]
gold_links = gold_links[gold_links.encOid.isin(GOLD)]
gold_reactions = gold_reactions[gold_reactions.encOid.isin(GOLD)]
assert set(gold_links.encOid) <= set(GOLD) and set(gold_context.encOid) <= set(GOLD)


def precision_recall_f1(true_positive, false_positive, false_negative):
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return round(precision, 3), round(recall, 3), round(f1, 3)


def span_scores(gold_spans, predicted_spans):
    """Exact-span precision, recall and F1 over two sets of (patient, start, end) keys."""
    return precision_recall_f1(len(gold_spans & predicted_spans),
                               len(predicted_spans - gold_spans),
                               len(gold_spans - predicted_spans))


print(f"held-out gold: {len(GOLD)} patients | discharge {len(gold_discharge)} | "
      f"admission {len(gold_admission)} | conditions {len(gold_context)} | "
      f"UMLS links {len(gold_links)} | reactions {len(gold_reactions)}")

### 7.1 Discharge extraction

The pipeline emits no character offsets for prescriptions, so a patient's extracted ingredients are
compared with the annotated ones as a **multiset** on the canonical key: a drug prescribed twice
counts twice.

In [ ]:
discharge_tp = discharge_fp = discharge_fn = 0
discharge_errors = []

for oid in GOLD:
    gold_counts = Counter(ingredient_key(name) for names in gold_discharge[gold_discharge.encOid == oid].active_ingredients
                          for name in names)
    predicted_counts = Counter(prescribed.loc[prescribed.encOid == oid, "ingredient_key"])

    for key in set(gold_counts) | set(predicted_counts):
        discharge_tp += min(gold_counts[key], predicted_counts[key])
        discharge_fp += max(0, predicted_counts[key] - gold_counts[key])
        discharge_fn += max(0, gold_counts[key] - predicted_counts[key])

    discharge_errors += [{"encOid": oid, "error": "false positive", "ingredient": k}
                         for k in set(predicted_counts) - set(gold_counts)]
    discharge_errors += [{"encOid": oid, "error": "missed", "ingredient": k}
                         for k in set(gold_counts) - set(predicted_counts)]

discharge_P, discharge_R, discharge_F1 = precision_recall_f1(discharge_tp, discharge_fp, discharge_fn)
print(f"discharge extraction: P {discharge_P} / R {discharge_R} / F1 {discharge_F1}  "
      f"(TP {discharge_tp}, FP {discharge_fp}, FN {discharge_fn})")
display(pd.DataFrame(discharge_errors))

### 7.2 Admission few-shot extraction

Exact character spans, because the gold annotates the mention as written and the grounding check of
Section 4 gives every prediction one.

In [ ]:
gold_admission_spans = {(int(r.encOid), int(r.mention_start), int(r.mention_end)) for r in gold_admission.itertuples()}
predicted_admission_spans = {(int(r.encOid), int(r.start), int(r.end))
                             for r in admission_mentions[admission_mentions.encOid.isin(GOLD)].itertuples()}
admission_P, admission_R, admission_F1 = span_scores(gold_admission_spans, predicted_admission_spans)

print(f"admission extraction: P {admission_P} / R {admission_R} / F1 {admission_F1}  "
      f"({len(predicted_admission_spans)} predicted, {len(gold_admission_spans)} annotated)")

admission_gold_text = {(int(r.encOid), int(r.mention_start), int(r.mention_end)): r.mention_text
                       for r in gold_admission.itertuples()}
admission_pred_text = {(int(r.encOid), int(r.start), int(r.end)): r.mention
                       for r in admission_mentions[admission_mentions.encOid.isin(GOLD)].itertuples()}
display(pd.DataFrame(
    [{"error": "spurious", "encOid": k[0], "offsets": f"{k[1]}:{k[2]}", "mention": admission_pred_text[k]}
     for k in sorted(predicted_admission_spans - gold_admission_spans)[:6]]
    + [{"error": "missed", "encOid": k[0], "offsets": f"{k[1]}:{k[2]}", "mention": admission_gold_text[k]}
       for k in sorted(gold_admission_spans - predicted_admission_spans)[:6]]))

### 7.3 Anamnesis condition NER

Two rows, two questions: did the tagger find the span at all, and did it also assign the right
category (`malattia` → `diagnosi`, `sintomo` → `sintomo`). The untyped row is an upper bound on the
typed one.

In [ ]:
CATEGORY_OF_LABEL = {"malattia": "diagnosi", "sintomo": "sintomo"}
predicted_conditions = (entities[entities.encOid.isin(GOLD) & entities.label.isin(CONDITION_LABELS)]
                        .sort_values("score", ascending=False)
                        .drop_duplicates(["encOid", "start", "end"]))   # a span under two labels is one prediction

gold_spans = {(int(r.encOid), int(r.start), int(r.end)) for r in gold_context.itertuples()}
predicted_spans = {(int(r.encOid), int(r.start), int(r.end)) for r in predicted_conditions.itertuples()}
gold_typed = {(*key, r.entity_type) for r, key in
              ((r, (int(r.encOid), int(r.start), int(r.end))) for r in gold_context.itertuples())}
predicted_typed = {(*key, CATEGORY_OF_LABEL[r.label]) for r, key in
                   ((r, (int(r.encOid), int(r.start), int(r.end))) for r in predicted_conditions.itertuples())}

ner_P, ner_R, ner_F1 = span_scores(gold_spans, predicted_spans)
typed_P, typed_R, typed_F1 = span_scores(gold_typed, predicted_typed)
display(pd.DataFrame([{"match": "span only", "P": ner_P, "R": ner_R, "F1": ner_F1},
                      {"match": "span and category", "P": typed_P, "R": typed_R, "F1": typed_F1}]).set_index("match"))
print(f"{len(predicted_spans)} predicted spans against {len(gold_spans)} annotated ones")

In [ ]:
# A few representative errors, with the sentence each came from.
gold_row_at = {(int(r.encOid), int(r.start), int(r.end)): r for r in gold_context.itertuples()}
pred_row_at = {(int(r.encOid), int(r.start), int(r.end)): r for r in predicted_conditions.itertuples()}

display(pd.DataFrame(
    [{"error": "spurious", "span": pred_row_at[k].text, "label": pred_row_at[k].label,
      "sentence": str(pred_row_at[k].sentence)[:70]} for k in sorted(predicted_spans - gold_spans)[:5]]
    + [{"error": "missed", "span": gold_row_at[k].condition, "label": gold_row_at[k].entity_type,
        "sentence": str(gold_row_at[k].sentence)[:70]} for k in sorted(gold_spans - predicted_spans)[:5]]))

### 7.4 Context classification

Macro-F1 per axis, against the majority-class baseline: with classes this imbalanced, a metric alone
is not interpretable, and `negated`, `possible` and `family` are exactly the cases a safety rule must
not get wrong. Predictions are made on the **gold spans**, so a NER miss is not charged to the
context model.

In [ ]:
context_true = {"assertion": [], "experiencer": []}
context_pred = {"assertion": [], "experiencer": []}

for oid, group in gold_context.groupby("encOid"):
    axes = context_axes(report_text(patient_by_oid[oid], T_ANAMNESIS), list(zip(group.start, group.end)))

    for axis in context_true:
        context_true[axis] += list(group[axis])
        context_pred[axis] += [a[axis] for a in axes]

context_scores = pd.DataFrame([
    {"axis": axis,
     "macro F1": round(f1_score(context_true[axis], context_pred[axis], average="macro", zero_division=0), 3),
     "majority baseline": round(f1_score(context_true[axis],
                                         [Counter(context_true[axis]).most_common(1)[0][0]] * len(context_true[axis]),
                                         average="macro", zero_division=0), 3),
     "mentions": len(context_true[axis])}
    for axis in context_true]).set_index("axis")
display(context_scores)

for axis in context_true:
    labels = sorted(set(context_true[axis]) | set(context_pred[axis]))
    print(f"{axis} (rows = gold, columns = predicted):")
    display(pd.DataFrame(confusion_matrix(context_true[axis], context_pred[axis], labels=labels),
                         index=labels, columns=labels))

### 7.5 UMLS entity linking

`gold_umls_links.csv` is a manually reviewed, mention-level **sample** of the held-out patients,
stratified over entity type and linking difficulty. A prediction is correct when its CUI is in the
annotated `acceptable_cuis` list, because UMLS often carries clinically equivalent concepts under
separate identifiers. Rows annotated as not linkable expect an abstention.

The floor and the margin were fixed before this file was read and nothing here changes them.

In [ ]:
linking_rows = []

for row in gold_links.itertuples():
    prediction = link_condition(str(row.mention), str(row.sentence))
    linked = prediction["link_status"] in UMLS_LINKED
    linking_rows.append({"mention": row.mention, "entity_type": row.entity_type,
                         "gold_linkable": row.gold_link_status == "linkable",
                         "gold_cui": row.gold_cui, "predicted_cui": prediction["umls_cui"],
                         "predicted_label": prediction["umls_label"],
                         "link_status": prediction["link_status"], "linked": linked,
                         "correct": bool(linked and prediction["umls_cui"] in set(row.acceptable_cuis))})

linking_eval = pd.DataFrame(linking_rows)
save_umls_cache()
linkable = linking_eval[linking_eval.gold_linkable]

umls_accuracy = round(linkable.correct.mean(), 3)
umls_coverage = round(linkable.linked.mean(), 3)
umls_nil_accuracy = round((~linking_eval.loc[~linking_eval.gold_linkable, "linked"]).mean(), 3)

display(pd.DataFrame([
    {"metric": "Accuracy@1", "value": umls_accuracy,
     "denominator": f"{len(linkable)} linkable mentions", "note": "an abstention counts as an error"},
    {"metric": "coverage", "value": umls_coverage,
     "denominator": f"{len(linkable)} linkable mentions", "note": "how often the linker committed at all"},
    {"metric": "correct abstention", "value": umls_nil_accuracy,
     "denominator": f"{int((~linking_eval.gold_linkable).sum())} gold-NIL mentions",
     "note": "declined where declining is right"},
]).set_index("metric"))

In [ ]:
def linking_outcome(row):
    """One outcome per gold mention, so the counts partition the sample."""
    if not row.gold_linkable:
        return "wrong link on a non-linkable mention" if row.linked else "correct abstention"

    if row.correct:
        return "correct link"

    return "wrong concept" if row.linked else "abstained on a linkable mention"


linking_eval["outcome"] = [linking_outcome(row) for row in linking_eval.itertuples()]
display(linking_eval.outcome.value_counts().rename("mentions").to_frame())
display(linking_eval[~linking_eval.correct & linking_eval.gold_linkable]
        [["mention", "gold_cui", "predicted_cui", "predicted_label", "link_status"]].head(6).reset_index(drop=True))

### 7.6 Alerts: annotated inputs against extracted inputs

There is no gold table of alerts, and there should not be: an alert is derived, not observed. What
can be measured is **error propagation**. The same four functions are run twice on the held-out
patients — once on inputs rebuilt from the annotation, once on the pipeline's own output — and the
two alert sets are compared. A difference is a difference in inputs, since the rules are identical.
The linker is the same in both runs, so what the comparison isolates is extraction and context error.

In [ ]:
# Run A inputs, rebuilt from the annotation.
gold_prescribed = (gold_discharge[gold_discharge.include_in_current_regimen.astype(bool)]
                   .explode("active_ingredients")
                   .rename(columns={"active_ingredients": "ingredient", "annotation_id": "prescription_id",
                                    "product_text": "product_name"})
                   .dropna(subset=["ingredient"])[["encOid", "prescription_id", "ingredient", "product_name"]]
                   .copy())
gold_prescribed["ingredient_key"] = gold_prescribed.ingredient.map(ingredient_key)
gold_prescribed["rxcui"] = gold_prescribed.ingredient_key.map(RXCUI).astype("Int64")
gold_prescribed["ddinter_id"] = gold_prescribed.ingredient_key.map(DDINTER_ID).astype("string")

# Gold condition mentions with their annotated context, linked by the SAME linker the pipeline uses.
gold_conditions = gold_context.rename(columns={"condition": "text"}).copy()
gold_conditions["usable"] = (gold_conditions.assertion == "affirmed") & (gold_conditions.experiencer == "patient")
gold_conditions = pd.concat(
    [gold_conditions.reset_index(drop=True),
     pd.DataFrame([link_condition(str(r.text), str(r.sentence)) for r in gold_conditions.itertuples()])], axis=1)
save_umls_cache()

# Gold reactions, on the same canonical key.
gold_reaction_rows = gold_reactions[(gold_reactions.assertion == "affirmed")
                                    & (gold_reactions.allergen_kind == "ingredient")].copy()
gold_reaction_rows["reaction_type"] = np.where(gold_reaction_rows.status == "intolerance", "intolerance", "allergy")
gold_reaction_rows["ingredient_keys"] = gold_reaction_rows.gold_key.map(
    lambda key: [ingredient_key(part) for part in str(key).split("/")])
gold_reaction_rows["mention"] = gold_reaction_rows.allergen_text
gold_reaction_rows["source"] = "gold"

print(f"run A inputs: {len(gold_prescribed)} annotated ingredient rows | "
      f"{int(gold_conditions.usable.sum())} usable annotated conditions | "
      f"{len(gold_reaction_rows)} annotated drug reactions")

In [ ]:
alerts_from_gold = run_alert_rules(gold_prescribed, gold_conditions, gold_reaction_rows)
alerts_from_pipeline = alerts[alerts.encOid.isin(GOLD)]


def alert_identity(row):
    """Patient, rule and the identifiers the rule fired on - never the printed name, which differs
    between an annotation and a parser for the same substance."""
    entities = {("drug", key) for key in (row.drug_a_key, row.drug_b_key) if pd.notna(key)}

    if pd.notna(row.condition_cui):
        entities.add(("condition", row.condition_cui))

    return row.encOid, row.alert_type, frozenset(entities)


identities_gold = {alert_identity(r) for r in alerts_from_gold.itertuples()}
identities_pipeline = {alert_identity(r) for r in alerts_from_pipeline.itertuples()}

alert_comparison = pd.DataFrame([
    {"alert type": alert_type,
     "annotated inputs": sum(1 for i in identities_gold if i[1] == alert_type),
     "pipeline inputs": sum(1 for i in identities_pipeline if i[1] == alert_type),
     "in both": sum(1 for i in identities_gold & identities_pipeline if i[1] == alert_type),
     "annotated only": sum(1 for i in identities_gold - identities_pipeline if i[1] == alert_type),
     "pipeline only": sum(1 for i in identities_pipeline - identities_gold if i[1] == alert_type)}
    for alert_type in sorted({i[1] for i in identities_gold | identities_pipeline})]).set_index("alert type")
display(alert_comparison)
print(f"total: {len(identities_gold)} alerts from annotated inputs, {len(identities_pipeline)} from "
      f"pipeline inputs, {len(identities_gold & identities_pipeline)} shared")

In [ ]:
# The disagreements themselves, so the propagation can be read rather than only counted.
def describe(identity):
    return ", ".join(f"{kind}={value}" for kind, value in sorted(identity[2]))


display(pd.DataFrame(
    [{"present in": side, "alert type": identity[1], "encOid": identity[0], "entities": describe(identity)}
     for side, difference in (("annotated only", identities_gold - identities_pipeline),
                              ("pipeline only", identities_pipeline - identities_gold))
     for identity in sorted(difference, key=str)[:5]]))

### 7.7 Scorecard

In [ ]:
scorecard = pd.DataFrame([
    ("discharge extraction", f"P/R/F1 {discharge_P}/{discharge_R}/{discharge_F1}",
     "ingredient multiset per patient, canonical key"),
    ("admission few-shot extraction", f"P/R/F1 {admission_P}/{admission_R}/{admission_F1}",
     "exact character spans"),
    ("anamnesis condition NER", f"P/R/F1 {ner_P}/{ner_R}/{ner_F1}",
     f"exact spans; with the category, F1 {typed_F1}"),
    ("context - assertion", f"macro-F1 {context_scores.loc['assertion', 'macro F1']}",
     f"majority baseline {context_scores.loc['assertion', 'majority baseline']}"),
    ("context - experiencer", f"macro-F1 {context_scores.loc['experiencer', 'macro F1']}",
     f"majority baseline {context_scores.loc['experiencer', 'majority baseline']}"),
    ("UMLS linking", f"Accuracy@1 {umls_accuracy}",
     f"coverage {umls_coverage}; correct abstention {umls_nil_accuracy}; "
     f"{len(linkable)} linkable mentions, single annotator"),
    ("alerts", f"{len(identities_gold & identities_pipeline)} of {len(identities_gold)} shared",
     "same rules on annotated and on extracted inputs; a difference is upstream error"),
], columns=["stage", "headline", "note"]).set_index("stage")

with pd.option_context("display.max_colwidth", None, "display.width", 200):
    display(scorecard)

**Reading the scorecard.** The numbers rank the stages by how much structure their source text
carries, not by how much effort went into them. Discharge extraction reads a filled form and scores
accordingly; anamnesis NER works on free prose against an exhaustive gold that annotates symptoms and
abnormal findings as conditions, and the residual gap is as much a policy difference as a model
limitation. Assertion and experiencer are the two axes Italian clinical writing marks with explicit
lexical cues, which is why a cue-based algorithm handles them. UMLS linking abstains rather than
guessing, so its coverage and its accuracy have to be read together. The alert comparison is the only
figure about the chain rather than a component: where the two runs differ is where an upstream error
changed what a rule was given.

## 8. RDF graph and SPARQL

**Task.** Put the structured outputs of every section into one queryable artifact.
**Method.** A small RDFS schema, one node per entity, and an ATC hierarchy declared with
`rdfs:subClassOf`.
**Output.** `knowledge_graph.ttl`, three SPARQL queries and one patient-level figure.
**Rationale.** Most of what the tables hold, a DataFrame answers just as well. The graph earns its
place on one thing: **hierarchy traversal**. Asking which of a patient's drugs are beta blockers
needs a walk up the ATC tree, and `rdfs:subClassOf*` does it without the query knowing which codes
those are.

| node | carries |
|---|---|
| `Patient` | the encounter identifier |
| `Prescription` | one discharge entry, with its dose text and its ATC class |
| `Ingredient` | one canonical ingredient, shared across patients |
| `ATCClass` | a code as a resource, linked to its parent by `rdfs:subClassOf` |
| `ClinicalMention` | one anamnesis mention, with its assertion and experiencer |
| `UMLSConcept` | the concept a mention denotes, when the linker committed |
| `Alert` | one rule firing, linked to the entities it involves |

A prescription is a node rather than a patient attribute, because the same ingredient can appear in
two of them — which is what rule 1 is about. A combination product stays one prescription with two
`containsIngredient` edges. A mention is neutral: it carries `assertion` and `experiencer`, and no
edge ever says the patient *has* a negated or family condition.

In [ ]:
EX = Namespace("http://example.org/clinical/")
ATC = Namespace("http://example.org/clinical/atc/")
UMLS = Namespace("http://example.org/clinical/umls/")

kg = Graph()
kg.bind("ex", EX)
kg.bind("atc", ATC)
kg.bind("umls", UMLS)

# TBox: the properties, with their domain and range. Alert subclasses are asserted so a query can ask
# by class; nothing entails them, because no reasoner runs.
SCHEMA = [(EX.hasPrescription, EX.Patient, EX.Prescription),
          (EX.containsIngredient, EX.Prescription, EX.Ingredient),
          (EX.atcClass, EX.Prescription, EX.ATCClass),
          (EX.hasMention, EX.Patient, EX.ClinicalMention),
          (EX.denotes, EX.ClinicalMention, EX.UMLSConcept),
          (EX.hasAlert, EX.Patient, EX.Alert),
          (EX.involvesDrug, EX.Alert, EX.Ingredient),
          (EX.involvesCondition, EX.Alert, EX.ClinicalMention)]

for prop, domain, range_ in SCHEMA:
    kg.add((prop, RDF.type, RDF.Property))
    kg.add((prop, RDFS.domain, domain))
    kg.add((prop, RDFS.range, range_))

ALERT_CLASS = {"duplicate_ingredient": EX.DuplicateIngredientAlert,
               "drug_drug_interaction": EX.DrugDrugInteractionAlert,
               "drug_allergy": EX.DrugReactionAlert,
               "drug_intolerance": EX.DrugReactionAlert,
               "drug_condition_contraindication": EX.DrugConditionAlert}

for alert_class in set(ALERT_CLASS.values()):
    kg.add((alert_class, RDFS.subClassOf, EX.Alert))

SCHEMA_SIZE = len(kg)
print(f"schema: {SCHEMA_SIZE} triples")

In [ ]:
# ATC codes are positional, so a code's ancestors are its prefixes: C07AB07 -> C07AB -> C07A -> C07 -> C.
ATC_LEVELS = [7, 5, 4, 3, 1]


def emit_atc(code):
    """Add the code and every ancestor it does not already have, and return the leaf URI."""
    chain = [ATC[str(code).strip().upper()[:n]] for n in ATC_LEVELS if len(str(code).strip()) >= n]

    for level, node in enumerate(chain):
        if (node, RDF.type, EX.ATCClass) not in kg:
            kg.add((node, RDF.type, EX.ATCClass))
            kg.add((node, RDFS.label, Literal(str(node).rsplit("/", 1)[-1])))

            if level + 1 < len(chain):
                kg.add((node, RDFS.subClassOf, chain[level + 1]))

    return chain[0] if chain else None


def slug(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_") or "x"

In [ ]:
# One pass over the patients. Each block adds the nodes of one entity type and the edges into it.
prescribed_by_patient = dict(list(prescribed.groupby("encOid")))
conditions_by_patient = dict(list(conditions_linked.groupby("encOid")))
alerts_by_patient = dict(list(alerts.groupby("encOid")))
node_counts = Counter()

for patient in patients:
    oid = patient["encOid"]
    patient_uri = EX[f"patient_{oid}"]
    kg.add((patient_uri, RDF.type, EX.Patient))
    kg.add((patient_uri, RDFS.label, Literal(str(oid))))
    node_counts["Patient"] += 1

    for prescription_id, rows in prescribed_by_patient.get(oid, prescribed.iloc[:0]).groupby("prescription_id"):
        first = rows.iloc[0]
        prescription_uri = EX[f"prescription_{prescription_id}"]
        kg.add((patient_uri, EX.hasPrescription, prescription_uri))
        kg.add((prescription_uri, RDF.type, EX.Prescription))
        node_counts["Prescription"] += 1

        for prop, value in (("productName", first.product_name), ("doseText", first.dose_text)):
            if pd.notna(value):
                kg.add((prescription_uri, EX[prop], Literal(str(value))))

        if pd.notna(first.atc_code):
            kg.add((prescription_uri, EX.atcClass, emit_atc(first.atc_code)))

        for row in rows.itertuples():
            ingredient_uri = EX[f"ingredient_{slug(row.ingredient_key)}"]
            kg.add((prescription_uri, EX.containsIngredient, ingredient_uri))

            if (ingredient_uri, RDF.type, EX.Ingredient) not in kg:
                kg.add((ingredient_uri, RDF.type, EX.Ingredient))
                kg.add((ingredient_uri, RDFS.label,
                        Literal(str(row.preferred_label if pd.notna(row.preferred_label) else row.ingredient))))
                node_counts["Ingredient"] += 1

                if pd.notna(row.rxcui):
                    kg.add((ingredient_uri, EX.rxcui, Literal(int(row.rxcui))))

    for row in conditions_by_patient.get(oid, conditions_linked.iloc[:0]).itertuples():
        mention_uri = EX[f"mention_{oid}_{int(row.start)}_{int(row.end)}"]
        kg.add((patient_uri, EX.hasMention, mention_uri))
        kg.add((mention_uri, RDF.type, EX.ClinicalMention))
        kg.add((mention_uri, RDFS.label, Literal(str(row.text))))
        kg.add((mention_uri, EX.assertion, Literal(row.assertion)))
        kg.add((mention_uri, EX.experiencer, Literal(row.experiencer)))
        node_counts["ClinicalMention"] += 1

        if row.link_status in UMLS_LINKED and pd.notna(row.umls_cui):
            concept_uri = UMLS[str(row.umls_cui)]
            kg.add((mention_uri, EX.denotes, concept_uri))

            if (concept_uri, RDF.type, EX.UMLSConcept) not in kg:
                kg.add((concept_uri, RDF.type, EX.UMLSConcept))
                kg.add((concept_uri, RDFS.label, Literal(str(row.umls_label))))
                node_counts["UMLSConcept"] += 1
        else:
            # The abstention is recorded rather than hidden, so a query for linked mentions cannot
            # silently pick this one up.
            kg.add((mention_uri, EX.linkStatus, Literal(str(row.link_status))))

    for position, row in enumerate(alerts_by_patient.get(oid, alerts.iloc[:0]).itertuples()):
        alert_uri = EX[f"alert_{oid}_{position}"]
        kg.add((patient_uri, EX.hasAlert, alert_uri))
        kg.add((alert_uri, RDF.type, ALERT_CLASS.get(row.alert_type, EX.Alert)))
        kg.add((alert_uri, RDF.type, EX.Alert))     # asserted too: no reasoner materialises it
        kg.add((alert_uri, EX.alertType, Literal(row.alert_type)))
        kg.add((alert_uri, EX.severity, Literal(str(row.severity))))
        kg.add((alert_uri, EX.source, Literal(str(row.source))))
        kg.add((alert_uri, RDFS.label, Literal(str(row.detail))))
        node_counts["Alert"] += 1

        for key in (row.drug_a_key, row.drug_b_key):
            if pd.notna(key):
                kg.add((alert_uri, EX.involvesDrug, EX[f"ingredient_{slug(key)}"]))

        if pd.notna(row.mention_id):
            kg.add((alert_uri, EX.involvesCondition, EX[f"mention_{row.mention_id}"]))

node_counts["ATCClass"] = len(set(kg.subjects(RDF.type, EX.ATCClass)))
kg.serialize(OUTPUT_DIR / "knowledge_graph.ttl", format="turtle")

print(f"knowledge graph: {len(kg):,} triples ({SCHEMA_SIZE} of them the schema) -> outputs/knowledge_graph.ttl")
display(pd.Series(dict(node_counts)).rename("nodes").to_frame())

### 8.1 Three queries

Q1 is a plain join. Q2 is the one the graph exists for. Q3 walks from an alert back to the entities
it was derived from.

In [ ]:
PREFIXES = ("PREFIX ex: <http://example.org/clinical/> PREFIX atc: <http://example.org/clinical/atc/> "
            "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>")


def run_query(title, feature, query, limit=8):
    print(f"\n### {title}\n    SPARQL feature: {feature}")
    rows = list(kg.query(PREFIXES + query))

    for row in rows[:limit]:
        print("   ", " | ".join("" if value is None else str(value).rsplit("/", 1)[-1] for value in row))

    if len(rows) > limit:
        print(f"    ... {len(rows) - limit} more rows")

    return rows


query_patient = int(alerts[alerts.alert_type == "drug_drug_interaction"].iloc[0].encOid)

run_query(f"Q1 - the discharge regimen of patient {query_patient}",
          "basic graph patterns with OPTIONAL",
          f"""SELECT ?product ?ingredient ?atc WHERE {{
                ex:patient_{query_patient} ex:hasPrescription ?prescription .
                ?prescription ex:containsIngredient ?i . ?i rdfs:label ?ingredient .
                OPTIONAL {{ ?prescription ex:productName ?product }}
                OPTIONAL {{ ?prescription ex:atcClass ?c . ?c rdfs:label ?atc }}
              }} ORDER BY ?ingredient""")

In [ ]:
# Q2: the property path, against the query that asks the same thing without one.
direct = list(kg.query(PREFIXES + """SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE {
                                       ?p ex:atcClass atc:C07 }"""))
traversed = list(kg.query(PREFIXES + """SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE {
                                          ?p ex:atcClass ?c . ?c rdfs:subClassOf* atc:C07 }"""))

print("### Q2 - prescriptions in ATC class C07 (beta blocking agents)\n"
      "    SPARQL feature: rdfs:subClassOf* property path")
display(pd.DataFrame([
    {"query": "?p ex:atcClass atc:C07", "prescriptions": int(direct[0][0]),
     "why": "no prescription carries the class directly; each carries a full 7-character code"},
    {"query": "?p ex:atcClass ?c . ?c rdfs:subClassOf* atc:C07", "prescriptions": int(traversed[0][0]),
     "why": "walks the hierarchy declared in the graph"}]).set_index("query"))

run_query("     the ingredients the path reaches", "property path, sample",
          """SELECT DISTINCT ?ingredient ?code WHERE {
               ?p ex:atcClass ?c ; ex:containsIngredient ?i . ?i rdfs:label ?ingredient .
               ?c rdfs:subClassOf* atc:C07 ; rdfs:label ?code .
             } ORDER BY ?code""", limit=6)

The membership is not stored anywhere: no triple says that this prescription belongs to `atc:C07`,
only that it belongs to `atc:C07AB07`, and that `C07AB07` is `rdfs:subClassOf` `C07AB`, which is
under `C07A`, which is under `C07`. The path retrieves the membership by walking those links at
query time.

Two things this is **not**. It is not OWL reasoning: no ontology, no reasoner, and `rdfs:subClassOf`
is used here as a plain hierarchy link. And rdflib materialises no entailments — the `subClassOf`
triples are inert data, and the same query without the `*` returns nothing, which is exactly what the
first row above shows.

In [ ]:
run_query("Q3 - the entities behind every drug-condition alert",
          "traversal from a derived alert back to its premises",
          """SELECT ?patient ?drug ?mention ?concept WHERE {
               ?p ex:hasAlert ?a . ?p rdfs:label ?patient .
               ?a a ex:DrugConditionAlert ; ex:involvesDrug ?d ; ex:involvesCondition ?m .
               ?d rdfs:label ?drug . ?m rdfs:label ?mention .
               OPTIONAL { ?m ex:denotes ?c . ?c rdfs:label ?concept }
             } ORDER BY ?patient""")

### 8.2 One patient, drawn from the graph

In [ ]:
# A deterministic, readable choice: among patients carrying a drug-condition alert - the shape where
# the whole chain is visible - the one with the fewest alerts and at most four prescriptions.
def alert_patients(alert_type):
    return set(alerts.loc[alerts.alert_type == alert_type, "encOid"])


small = {oid for oid, rows in prescribed.groupby("encOid") if rows.prescription_id.nunique() <= 4}
figure_patient = min(sorted(alert_patients("drug_condition_contraindication") & small),
                     key=lambda oid: (int((alerts.encOid == oid).sum()), oid))

NODE_STYLE = {EX.Patient: (0, "#34495E"), EX.Prescription: (1, "#7F8C8D"), EX.ClinicalMention: (1, "#8E44AD"),
              EX.Ingredient: (2, "#2980B9"), EX.UMLSConcept: (2, "#16A085"), EX.ATCClass: (3, "#16A085"),
              EX.Alert: (4, "#C0392B")}


def subgraph_of(root, depth=3):
    """Every triple reachable from `root` within `depth` hops."""
    reachable = Graph()
    frontier, seen = {root}, set()

    for _ in range(depth + 1):
        next_frontier = set()

        for node in frontier - seen:
            seen.add(node)

            for prop, obj in kg.predicate_objects(node):
                reachable.add((node, prop, obj))

                if isinstance(obj, URIRef):
                    next_frontier.add(obj)

        frontier = next_frontier

    return reachable


def label_of(graph, node):
    label = graph.value(node, RDFS.label)
    return textwrap.fill(str(label if label is not None else str(node).rsplit("/", 1)[-1])[:38], 16)


patient_graph = subgraph_of(EX[f"patient_{figure_patient}"])
figure = nx.MultiDiGraph()

for subject, prop, obj in patient_graph:
    if prop in (RDF.type, RDFS.label) or not isinstance(obj, URIRef):
        continue

    for node in (subject, obj):
        if node not in figure:
            layer, colour = NODE_STYLE.get(patient_graph.value(node, RDF.type), (2, "#BDC3C7"))
            figure.add_node(node, layer=layer, colour=colour, label=label_of(patient_graph, node))

    figure.add_edge(subject, obj, key=str(prop), relation=str(prop).rsplit("/", 1)[-1])

positions = nx.multipartite_layout(figure, subset_key="layer")
fig, ax = plt.subplots(figsize=(13, 7))
nx.draw_networkx_nodes(figure, positions, ax=ax, node_size=1700,
                       node_color=[figure.nodes[n]["colour"] for n in figure], alpha=0.92)
nx.draw_networkx_labels(figure, positions, ax=ax, font_size=6, font_color="white",
                        labels={n: figure.nodes[n]["label"] for n in figure})
nx.draw_networkx_edges(figure, positions, ax=ax, edge_color="#95A5A6", arrowsize=9, width=1.0,
                       connectionstyle="arc3,rad=0.06")
nx.draw_networkx_edge_labels(figure, positions, ax=ax, font_size=5,
                             edge_labels={(u, v): d["relation"] for u, v, d in figure.edges(data=True)})
ax.set_title(f"patient {figure_patient}: prescriptions, mentions, concepts, ATC classes and the alerts "
             f"derived from them", fontsize=10)
ax.axis("off")
plt.tight_layout()
plt.show()

**Output of Section 8.** Every table produced above converges into one artifact: patients,
prescriptions and their ingredients, ATC classes arranged by `rdfs:subClassOf`, anamnesis mentions
with their context and either a concept or a recorded abstention, and alerts as nodes that point back
at the entities they were derived from. The queries that are not Q2 could be pandas aggregations over
the same tables; Q2 could not, without a hand-written member list or a join per level.

## 9. Local LLM demonstrations

**Task.** Show three ways a small local model can sit next to a symbolic system.
**Method.** `qwen2.5:3b-instruct` through Ollama, at temperature 0, using the same `generate_local`
helper as Section 4.
**Output.** Three worked demonstrations with diagnostic counts.
**Rationale.** These are demonstrations of a mechanism, not benchmarks. Nothing here is scored
against a clinical reference, and no clinical fact originates in the model: the alerts were already
computed in Section 6.

The patient used below belongs to the cohort already processed above. These demonstrations show the
path end to end; they say nothing about unseen input.

1. **Grounded verbalisation** — the model receives facts that are already computed and puts them into
   Italian. It decides nothing.
2. **Tool calling over the graph** — the model chooses which symbolic operation to invoke; each tool
   runs a query written and validated here.
3. **Direct SPARQL generation** — the same model asked to write the query itself, checked separately
   for format, validity and correctness. It is what motivates part 2.

### 9.1 Grounded verbalisation

In [ ]:
# A real cohort patient, chosen deterministically: the lowest identifier carrying a Major interaction.
demo_patient = int(sorted(alerts.loc[(alerts.alert_type == "drug_drug_interaction")
                                     & (alerts.severity == "Major"), "encOid"].unique())[0])

GROUNDING_RULES = ("Rispondi SOLO con i fatti forniti. Non aggiungere meccanismi farmacologici, "
                   "raccomandazioni o interpretazioni cliniche. Se i fatti non bastano, dillo. "
                   "Massimo 120 parole.")


def facts_for(oid):
    """The structured rows of one patient, unparsed into the plain text that goes into the prompt."""
    drugs = sorted(set(prescribed.loc[prescribed.encOid == oid, "ingredient"].astype(str)))
    usable = conditions_linked[(conditions_linked.encOid == oid) & conditions_linked.usable]
    lines = ["FARMACI ALLA DIMISSIONE: " + ", ".join(drugs),
             "CONDIZIONI AFFERMATE: " + ", ".join(sorted(set(usable.text.astype(str)))[:12])]

    return "\n".join(lines + [f"ALERT [{r.alert_type} | fonte {r.source}]: {r.detail}"
                              for r in alerts[alerts.encOid == oid].itertuples()])


demo_facts = facts_for(demo_patient)
demo_question = "Riassumi le segnalazioni di sicurezza per questo paziente e indica su quali fatti si basano."

print(f"patient {demo_patient}\n")
print("[system]", GROUNDING_RULES)
print("\n[user] FATTI:\n" + demo_facts)
print("\n[user] DOMANDA:", demo_question)

demo_answer = generate_local(f"FATTI:\n{demo_facts}\n\nDOMANDA: {demo_question}",
                             system=GROUNDING_RULES)["content"].strip()
print("\n[model]\n" + wrap(demo_answer))

In [ ]:
# A lexical grounding check, and nothing more: did the answer name a substance or a severity the
# facts do not contain? It is a membership test over two closed vocabularies. It cannot see an
# invented mechanism, an inverted negation, a relation asserted between the wrong entities, or an
# omission - so it bounds one failure mode and says nothing about the others.
answer_lower = demo_answer.lower()
fact_ingredients = set(prescribed.loc[prescribed.encOid == demo_patient, "ingredient_key"])
named_ingredients = {key for key in PREFERRED_LABEL
                     if len(key) >= 6 and re.search(rf"\b{re.escape(key)}\b", answer_lower)}
fact_severities = {str(s).lower() for s in alerts.loc[alerts.encOid == demo_patient, "severity"]}
named_severities = {word for word in ("major", "moderate", "minor", "grave", "severa", "critica")
                    if word in answer_lower}

print("drugs named but absent from the facts:     ", sorted(named_ingredients - fact_ingredients) or "(none)")
print("severities named but absent from the facts:", sorted(named_severities - fact_severities) or "(none)")

### 9.2 Tool calling over the graph

The model never writes a query. It picks a tool and its arguments; each tool runs SPARQL written
here. One of the four questions has no applicable tool — the graph holds no vital signs — and the
only correct behaviour there is to call nothing.

In [ ]:
def get_patient_regimen(patient_id):
    """The active ingredients prescribed to one patient at discharge."""
    rows = kg.query(PREFIXES + f"""SELECT DISTINCT ?ingredient WHERE {{
                                     ex:patient_{patient_id} ex:hasPrescription ?p .
                                     ?p ex:containsIngredient ?i . ?i rdfs:label ?ingredient }}""")
    return sorted(str(r[0]) for r in rows) or ["(nessun farmaco)"]


def get_patient_alerts(patient_id):
    """Every alert raised for one patient, as the rule described it."""
    rows = kg.query(PREFIXES + f"""SELECT ?type ?detail WHERE {{
                                     ex:patient_{patient_id} ex:hasAlert ?a .
                                     ?a ex:alertType ?type ; rdfs:label ?detail }}""")
    return [f"{r[0]}: {r[1]}" for r in rows] or ["(nessun alert)"]


def get_drugs_in_atc_class(patient_id, atc_prefix):
    """The patient's drugs under an ATC class, following the hierarchy."""
    rows = kg.query(PREFIXES + f"""SELECT DISTINCT ?ingredient ?code WHERE {{
                                     ex:patient_{patient_id} ex:hasPrescription ?p .
                                     ?p ex:containsIngredient ?i ; ex:atcClass ?c .
                                     ?i rdfs:label ?ingredient .
                                     ?c rdfs:subClassOf* atc:{atc_prefix} ; rdfs:label ?code }}""")
    return [f"{r[0]} ({r[1]})" for r in rows] or [f"(nessun farmaco sotto {atc_prefix})"]


TOOLS = {"get_patient_regimen": get_patient_regimen,
         "get_patient_alerts": get_patient_alerts,
         "get_drugs_in_atc_class": get_drugs_in_atc_class}

TOOL_SPEC = [
    {"type": "function", "function": {
        "name": "get_patient_regimen",
        "description": "Elenca i principi attivi prescritti alla dimissione per un paziente.",
        "parameters": {"type": "object", "properties": {"patient_id": {"type": "string"}},
                       "required": ["patient_id"]}}},
    {"type": "function", "function": {
        "name": "get_patient_alerts",
        "description": "Elenca le segnalazioni di sicurezza di un paziente.",
        "parameters": {"type": "object", "properties": {"patient_id": {"type": "string"}},
                       "required": ["patient_id"]}}},
    {"type": "function", "function": {
        "name": "get_drugs_in_atc_class",
        "description": "Elenca i farmaci del paziente appartenenti a una classe ATC, seguendo la gerarchia "
                       "(C07 = betabloccanti, C10AA = statine, J01C = penicilline).",
        "parameters": {"type": "object", "properties": {"patient_id": {"type": "string"},
                                                        "atc_prefix": {"type": "string"}},
                       "required": ["patient_id", "atc_prefix"]}}},
]

EXPECTED_TOOL = {
    f"Il paziente {demo_patient} assume betabloccanti?": "get_drugs_in_atc_class",
    f"Quali farmaci ha alla dimissione il paziente {demo_patient}?": "get_patient_regimen",
    f"Che segnalazioni di sicurezza ha il paziente {demo_patient}?": "get_patient_alerts",
    f"Qual e la pressione arteriosa del paziente {demo_patient}?": None,
}

In [ ]:
ROUTING_RULES = "Scegli lo strumento adatto. Non inventare dati. Se nessuno strumento puo rispondere, dillo."
trace = []

for question, expected in EXPECTED_TOOL.items():
    print("\n" + "-" * 90 + f"\nQ: {question}")
    message = generate_local(question, system=ROUTING_RULES, tools=TOOL_SPEC)
    calls = message.get("tool_calls") or []

    if not calls:
        print("   [no tool selected]", message["content"].strip()[:180])
        trace.append({"question": question[:48], "expected": expected or "(none)", "selected": "(none)",
                      "routed correctly": expected is None, "answer uses the facts": None})
        continue

    name = calls[0]["function"]["name"]
    arguments = dict(calls[0]["function"].get("arguments") or {})
    returned = TOOLS[name](**arguments) if name in TOOLS else ["(strumento inesistente)"]
    print(f"   [tool] {name}({arguments})\n   [facts] {str(returned)[:180]}")

    answer = generate_local(f"FATTI: {returned}\n\nDOMANDA: {question}",
                            system=GROUNDING_RULES)["content"].strip()
    print("   [answer]", answer[:240])
    # A lexical proxy for "the answer used what the tool returned": it repeats a substantive word
    # from the facts. It does not judge whether the answer is right.
    vocabulary = {w.lower() for fact in returned for w in re.findall(r"[A-Za-zÀ-ÿ]{6,}", str(fact))}
    trace.append({"question": question[:48], "expected": expected or "(none)", "selected": name,
                  "routed correctly": name == expected and str(demo_patient) in str(arguments),
                  "answer uses the facts": any(w in answer.lower() for w in vocabulary)})

routing = pd.DataFrame(trace)
display(routing)
print(f"routing: {int(routing['routed correctly'].sum())}/{len(routing)} questions sent to the expected "
      f"tool (the unanswerable one counts only when no tool is called)")
print(f"use of the returned facts: {int(routing['answer uses the facts'].fillna(False).sum())}/"
      f"{int(routing['answer uses the facts'].notna().sum())} answers repeat one")

### 9.3 The same model asked to write SPARQL

Four checks grouped into three questions, because they fail differently: did the output obey the
requested format, is the query it contains valid (does it parse, does it execute), and does it answer
what was asked.

In [ ]:
SCHEMA_DESCRIPTION = ("Classi: ex:Patient, ex:Prescription, ex:Ingredient, ex:ATCClass, ex:Alert. "
                      "Proprieta: ex:hasPrescription, ex:containsIngredient, ex:atcClass, ex:hasAlert, "
                      "ex:alertType, rdfs:label. Prefisso ex: <http://example.org/clinical/>.")

raw_answer = generate_local(f"Schema RDF: {SCHEMA_DESCRIPTION}\nScrivi SOLO una query SPARQL che elenchi "
                            f"i principi attivi del paziente ex:patient_{demo_patient}.")["content"]
print("model output, verbatim:\n")
print(raw_answer[:700])

fenced = re.search(r"```(?:sparql)?\s*(.*?)```", raw_answer, flags=re.DOTALL | re.IGNORECASE)
query_text = fenced.group(1).strip() if fenced else raw_answer.strip()
obeyed_format = query_text == raw_answer.strip()

parses = executes = False
failure = ""

try:
    prepareQuery(query_text)
    parses = True
except Exception as error:              # rdflib reports every syntax problem as an exception
    failure = f"parse: {type(error).__name__}: {str(error)[:110]}"

if parses:
    try:
        returned = list(kg.query(query_text))
        executes = True
    except Exception as error:
        returned = []
        failure = f"execution: {type(error).__name__}: {str(error)[:110]}"
else:
    returned = []

expected_rows = list(kg.query(PREFIXES + f"""SELECT ?i ?label WHERE {{
                                               ex:patient_{demo_patient} ex:hasPrescription ?p .
                                               ?p ex:containsIngredient ?i . ?i rdfs:label ?label }}"""))
expected_uris = {str(r[0]) for r in expected_rows}
expected_labels = {str(r[1]) for r in expected_rows}
returned_values = {str(value) for row in returned for value in row if value is not None}
# The query answers the question when what it returns IS the patient's ingredients, as node URIs
# or as labels. A subset would be incomplete and a superset would be another patient's drugs too.
answers_question = bool(expected_uris) and returned_values in (expected_uris, expected_labels)

display(pd.DataFrame([
    {"check": "the output is a query and nothing else", "result": obeyed_format},
    {"check": "the extracted query parses", "result": parses},
    {"check": "it executes against this graph", "result": executes},
    {"check": "it returns the patient's ingredients", "result": answers_question},
]).set_index("check"))

if failure:
    print(failure)
else:
    print(f"rows returned: {len(returned)} | expected ingredients: {len(expected_uris)}")

**Reading part 3.** The four checks are deliberately separate. Wrapping a query in an explanation is
a **formatting** failure and says nothing about SPARQL; whether the query inside the fence parses,
executes and returns the right rows is the question that does. Whatever this run produced, the
argument for part 2 is narrower than "the model cannot write queries": a tool call has a typed
signature, so the format is not something the model has to get right, and the query it triggers was
validated before the model ever saw the question.

## 10. Discussion

**Method follows surface form.** The three reports needed three techniques, and the reason is the
text, not the ambition. *Terapia alla Dimissione* is a filled template, so a regex reads it and
scores near the ceiling. *Terapia medica all'ingresso* has no stable shape, so a compact few-shot
extractor covers its variants with one prompt; Section 7.2 says what that costs in exact-span terms.
*Anamnesi* is free prose with no enumerable lexicon, so it needs zero-shot NER, a context algorithm
and an entity linker, and it is the least accurate stage of the three.

**What the context model does and does not model.** Assertion and experiencer are kept because
Italian clinical writing marks them with explicit lexical cues that ConText's sentence scope can
reach, and Section 7.4 shows both beating their majority baseline. Whether a condition is current or
historical is not modelled: no rule reads it, so extracting it would add a stage nothing consumes.

**Dependency parsing was explored, not adopted.** Section 5.3 parses two real sentences and reports
whether the negation cue governs each condition. Coordination is where a general-purpose Italian
parse stops being a reliable basis for scope in this register, so the operational pipeline stays
parser-free. No F1 is claimed for the alternative.

**The alerts are new facts.** Each one is derived from premises that live in different places — two
reports and an external table — and is written in none of them. That is the contribution of the
symbolic layer, and it is exactly why the layer cannot be better than its inputs.

**Long chains break first.** Duplicate-ingredient and interaction alerts join on identifiers the
discharge template states almost verbatim. The drug-condition rule sits behind two linking steps: an
Italian phrase to a UMLS concept, and a MeSH descriptor to the same vocabulary. Section 7.6 runs the
same four functions on annotated and on extracted inputs, and its table is where an upstream error
becomes a different alert. Keeping that rule despite its weakness is the point: it is the one place
where knowledge integration and error propagation are both visible.

**What RDF adds.** One artifact instead of a folder of tables, and one capability they do not have:
`rdfs:subClassOf*` retrieves ATC class membership that no triple states. That is SPARQL property-path
traversal over a declared hierarchy — not OWL reasoning, and not rdflib materialising entailments.

**Where the LLM sits.** It extracts admission mentions, verbalises facts that are already computed,
and chooses which validated query to run. It is never the source of a clinical assertion: every
interaction comes from DDInter, every contraindication from MED-RT, every concept identity from UMLS.

**Limits.** A single annotator, so no agreement ceiling. Thirty held-out patients, so small
differences are not resolvable. Rule recall is bounded by identifier coverage — the shares printed in
Section 3 — and agreeing with a knowledge base is faithful reproduction of that base, not clinical
validity. Prescriptions written outside the quoted discharge block are not extracted. No prescription
status is derived, so the eight entries whose instructions describe a suspension still reach the
rules; the instruction text is kept, but nothing reads it. Allergen mentions that name a class rather
than a substance stay unresolved by design. None of the linking output is suitable for clinical use.